# MagiNet 기반 TrafficFlowBench Task 1·3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mmt-1234/trafficflowbench/blob/main/notebooks/task13_maginet_colab.ipynb)

**MagiNet의 AMSTenc와 MASTdec를 이 파일에 직접 구현했습니다.** 데이터 처리, 공동 학습, 체크포인트, 평가, 보간 대비 선택, Task 1 예측 CSV까지 노트북 안에 있습니다. 별도 `.py`, 프로젝트 체크아웃, 모델 다운로드가 필요하지 않습니다. 대회 데이터와 아래 설치 패키지는 필요합니다.

```text
채널별 공개 관측 + 관측 마스크
→ AMSTenc: 관측 임베딩 / 학습 가능한 결측 토큰 / 시간·도로 보조 임베딩
→ MASTdec × 3: 마스크 인지 시간·공간 Attention
              + 마스크 인지 Chebyshev 그래프 합성곱
              + 3·5·7 다중 시간 게이트
→ 블록 출력 결합 → 시간축 양방향 GRU → 속도 보정량 / 차로당 교통량 보정량
→ 공개 관측으로 만든 보간값 + 보정량 (음수 결과는 0으로 제한)
→ Task 1 관측소별 복원 → 링크별 N·ΔN·FD 보조 손실
```

논문 및 공식 구현을 참고한 **대회 적응 재구현**입니다. 원본의 단일 관측 채널·고정 노드 수·12-step 시간층·출력층을 확장했습니다. 채널별 missing token, 명확한 관측/결측 마스크, 모든 키가 결측인 Attention 처리, 정규화 Laplacian과 행렬곱 Chebyshev 재귀, 가변 노드 수, 양방향 시간 readout을 사용합니다. 원논문 점수 재현이라고 주장하지 않습니다.

관측소를 노드로 유지하고 패널별 도로 그래프에서 공통 가중치를 학습합니다. 도로군·N/S/E/W·관측소 임베딩을 사용합니다. 코리더 간 직접 관측 교환은 넣지 않습니다. 실제 관측/결측 토큰 경로를 유지하며 보간값은 별도 보조 임베딩과 잔차 출력의 기준으로 사용합니다. 입력 표준화는 fit 구간의 관측만 사용하며 출력·손실·물리 계산의 /25·/600 단위는 유지합니다.

**실제 데이터 학습·점수·GPU 메모리·실행 시간은 아직 측정하지 않았습니다.** Task 3의 N·ΔN·FD 진단은 공개 train에서 계산하는 대리 지표이며 공식 숨은 경계 flux를 재현하지 않습니다. Task 3는 별도 제출 없이 Task 1 예측을 재사용합니다.

## 1. 설치·Drive·설정

Colab의 GPU 런타임에서 위에서부터 실행해 주세요. PyTorch는 Colab 설치 버전을 사용합니다. 로컬 실행에서는 PyTorch 2.3 이상이 필요합니다. `RELEASE_ROOT`에는 이미 받은 대회 데이터의 `corridors/`와 `task1/`이 있어야 합니다. 데이터를 다시 다운로드하지 않습니다.

GitHub와 Colab 사이의 코드 저장 순서는 다음과 같습니다.

1. 로컬 수정본을 GitHub에 저장한 뒤 위의 **Open In Colab** 버튼으로 최신본을 엽니다.
2. Colab에서 코드를 수정했다면 **파일 → GitHub에 사본 저장**에서 `mmt-1234/trafficflowbench`, `main`, `notebooks/task13_maginet_colab.ipynb`를 선택합니다.
3. 로컬에서 다시 수정하기 전에 GitHub의 최신본을 먼저 가져옵니다. 자동 동기화는 아니며 데이터·체크포인트·결과 파일은 Drive에 유지합니다.

비공개 저장소를 처음 열 때는 [Colab의 GitHub 탭](https://colab.research.google.com/github)에서 **Include Private Repos**를 선택하고 GitHub 접근을 연결해 주세요.

| 구간 | 역할 |
| --- | --- |
| 2030-06~12 | 학습, 보간·입력 특징·출력 초기화용 과거 평균 |
| 2031-01-01~15 | 체크포인트 조기 종료 |
| 2031-01-16~31 | MagiNet과 보간의 공통 혼합 비율 선택 |
| 2031-02 | 선택 비율을 고정한 추가 진단, 기본 생략 |

1월 두 구간은 모두 모델 선택용입니다. 이전 실험에 사용한 2월도 독립적인 최종 검증으로 보지 않습니다. validation/private 정답은 읽지 않습니다. 설정은 검증 전 시작값이며 비교 실험에는 `REPORT_DIR`를 다르게 지정해 주세요.

이번 개선판은 **보간 잔차 출력 + 입력 표준화**입니다. `INPUT_NORMALIZATION='standard'`가 기본이며 `'fixed'`로 기존 고정 스케일과 비교할 수 있습니다. 선택 proxy·물리 보조 손실은 유지합니다. 새 출력 구조·표준화 버퍼가 있으므로 이전 절대값 모델 체크포인트는 재개하지 않습니다. 새 결과 폴더를 기본으로 사용하며 같은 구조·표준화의 실행만 `RESUME=True`로 재개합니다.

저장 입력은 `SAVED_CACHE_ZIP`의 `preprocessed-86qnwmwe.zip`을 `CACHE_DIR`에 한 번 풀어 사용합니다. 같은 ZIP을 다시 실행하면 로컬 복원본을 재사용합니다. ZIP 내부는 `<패널>/fit|early|blend/YYYY-MM-DD.npz` 구조여야 합니다. 이미 푼 폴더를 사용할 때는 `SAVED_CACHE_ZIP=None`으로 하고 `CACHE_DIR`를 그 폴더로 지정해 주세요.

기존 NPZ에는 패널 그래프·fit 과거 평균의 전체 정보가 없으므로 **원래 데이터의 network·템플릿·관측소 구성과 이전 `full/config.json`, `full/fit_profile-<패널>.npy`도 필요합니다.** 이전 결과의 `full/` 폴더를 `CACHE_PROFILE_DIR`로 지정해 주세요. ZIP 안에 함께 들어 있으면 자동으로 찾습니다. 과거 평균을 한 번 불러온 후에는 캐시 폴더에 저장해 재사용합니다.

`CACHE_MODE='reuse'`에서는 캐시·과거 평균 누락 또는 기간/대상 불일치 시 중단하며 원본을 다시 전처리하지 않습니다. 처음부터 만들거나 누락 날짜를 채울 때만 `'build'`를 사용합니다. 입력 표준화 통계는 fit의 관측값을 대상으로 한 번 계산해 Drive 실행 폴더에 저장하며 정답·early·blend·2월은 포함하지 않습니다. NPZ 내부 values/visible 배열의 CRC와 fit 날짜로 재사용을 확인하므로 같은 입력을 다시 풀어도 통계를 재계산하지 않습니다. 이 통계 순회와 ZIP 복원에는 시간이 필요하지만 보간·램프 특징을 다시 생성하지 않습니다.


In [ ]:
%pip -q install "pandas>=2.0" numpy pyarrow matplotlib


In [ ]:
try:
    from google.colab import drive
except ImportError:
    print('로컬 실행: RELEASE_ROOT를 실제 데이터 경로로 설정해 주세요.')
else:
    drive.mount('/content/drive')


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from functools import lru_cache
import hashlib, heapq, json, random, re, shutil, tempfile, time, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

RELEASE_ROOT = Path('/content/drive/MyDrive/dataset/trafficflow')
INPUT_NORMALIZATION = 'standard'  # 'fixed': 기존 /25·/600, 'standard': fit 관측으로 입력 표준화
REPORT_DIR = RELEASE_ROOT.parent / f'trafficflow_maginet_residual_{INPUT_NORMALIZATION}_reports'
SAVED_CACHE_ZIP = Path('/content/drive/MyDrive/dataset/trafficflow/trafficflow_maginet_cache/preprocessed-86qnwmwe.zip')
CACHE_DIR = Path('/content/trafficflow_maginet_cache_restored')  # ZIP을 한 번 풀 로컬 폴더
CACHE_MODE = 'reuse'             # 'reuse': 누락 시 중단, 'build': 없는 날짜만 원본에서 생성
CACHE_PROFILE_DIR = RELEASE_ROOT.parent / 'trafficflow_maginet_reports' / 'full'  # 이전 config.json·fit_profile-*.npy
PANELS = None                    # None: 전체 패널 공동 학습
MAX_TRAIN_DAYS = None            # 실행 점검 시 12 이상으로 줄일 수 있습니다.
SEED = 42
WINDOW, STRIDE = 72, 36
WIDTH, BLOCKS, HEADS, CHEB_K = 64, 3, 4, 3
GRAPH_DIM = 128
DROPOUT = 0.05
MGL_THRESHOLD = 0.5              # 공식 구현의 missing graph 판정 시작값
BATCH_SIZE = 2
UPDATES_PER_GROUP = 24           # epoch당 패널×regime 업데이트
EPOCHS, PATIENCE = 50, 8
LR, WEIGHT_DECAY = 1e-3, 1e-4
WARMUP_EPOCHS, PHYSICS_RAMP_EPOCHS = 5, 5
LOSS_WEIGHTS = dict(N=0.05, dN=0.10, FD=0.0)
# 공식 Task 1·3 비중을 참고한 대리 선택식입니다. 공식 Physics 점수가 아닙니다.
SELECT_WEIGHTS = dict(State=0.70, FD_proxy=0.10, LWR_proxy=0.20)
ALPHAS = (0.0, 0.25, 0.5, 0.75, 1.0)
RUN_FIRST_PANEL = True
FIRST_EPOCHS, FIRST_DAYS_PER_REGIME = 2, 4
RESUME = False                   # last_maginet.pt에서 재개; EPOCHS를 마지막 epoch보다 크게 설정
RUN_FEBRUARY = False
PREDICT_SPLIT = 'validation'     # validation, private 또는 None
LWR_MODES = {}                   # 공개 기본값 A; 패널별 B/C 실험 시 명시
N_JOBS = 4

REGIMES = ('R1', 'R2', 'R3')
KEYS = ['panel', 'timestamp', 'station_id', 'link_id', 'mask_regime']
SCALE = np.array([25., 600.], dtype=np.float32)
PERIODS = dict(fit=('2030-06-01', '2030-12-31'), early=('2031-01-01', '2031-01-15'),
               blend=('2031-01-16', '2031-01-31'), february=('2031-02-01', '2031-02-28'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
assert tuple(map(int, re.match(r'(\d+)\.(\d+)', torch.__version__).groups())) >= (2, 3)
assert 12 <= WINDOW <= 288 and 1 <= STRIDE <= WINDOW
assert WIDTH % HEADS == 0 and WIDTH % 2 == 0 and CHEB_K >= 1 and BLOCKS >= 1
assert BATCH_SIZE >= 1 and UPDATES_PER_GROUP >= 1 and EPOCHS >= 1
assert PREDICT_SPLIT in (None, 'validation', 'private')
assert INPUT_NORMALIZATION in ('fixed', 'standard') and CACHE_MODE in ('reuse', 'build')
assert all(v >= 0 for v in LOSS_WEIGHTS.values()) and sum(SELECT_WEIGHTS.values()) > 0
assert all(v >= 0 for v in SELECT_WEIGHTS.values())
assert WARMUP_EPOCHS >= 0 and PHYSICS_RAMP_EPOCHS >= 0
assert len(set(ALPHAS)) == len(ALPHAS) and all(0 <= a <= 1 for a in ALPHAS) and 0.0 in ALPHAS and 1.0 in ALPHAS

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_seed()
torch.set_num_threads(N_JOBS)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print({'device': str(DEVICE), 'amp': USE_AMP, 'torch': str(torch.__version__), 'reports': str(REPORT_DIR)})


## 2. 데이터 계약과 패널 준비

공개 masked 파일을 입력으로, train의 original 파일을 정답으로 사용합니다. 템플릿에 지정된 셀만 학습·평가합니다. 대상 정답이 입력에 보이거나 중복 키, station/link 또는 regime 불일치가 있으면 처리를 중단합니다. 내부 교통량은 차로당 값이며 CSV의 `flow_vph`는 관측소의 합계입니다.

In [ ]:
def scan_layout(link_ids: list[str], topology: pd.DataFrame) -> list[str]:
    """Topological order within each weakly connected component; cycles are stable."""
    links = set(link_ids)
    outgoing = {link: set() for link in links}
    neighbors = {link: set() for link in links}
    for row in topology.itertuples(index=False):
        target = str(row.link_id)
        if target not in links:
            continue
        for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
            source = source.strip()
            if source in links and source != target:
                outgoing[source].add(target)
                neighbors[source].add(target)
                neighbors[target].add(source)
        for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
            dest = dest.strip()
            if dest in links and dest != target:
                outgoing[target].add(dest)
                neighbors[target].add(dest)
                neighbors[dest].add(target)
    seen: set[str] = set()
    ordered: list[str] = []
    for root in sorted(links):
        if root in seen:
            continue
        component = {root}
        pending = [root]
        seen.add(root)
        while pending:
            for neighbor in neighbors[pending.pop()]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    component.add(neighbor)
                    pending.append(neighbor)
        indegree = {link: 0 for link in component}
        for source in component:
            for target in outgoing[source] & component:
                indegree[target] += 1
        ready = [link for link in component if indegree[link] == 0]
        heapq.heapify(ready)
        part = []
        while ready:
            source = heapq.heappop(ready)
            part.append(source)
            for target in outgoing[source] & component:
                indegree[target] -= 1
                if indegree[target] == 0:
                    heapq.heappush(ready, target)
        part.extend(sorted(component - set(part)))
        ordered.extend(part)
    return ordered

@dataclass
class PanelInfo:
    name: str
    links: list[str]
    link_index: dict[str, int]
    free_speed: np.ndarray
    lanes: np.ndarray
    lengths: np.ndarray
    station_lanes: dict[str, float]
    capacity: np.ndarray | None = None
    jam_density: np.ndarray | None = None
    edges: np.ndarray = field(default_factory=lambda: np.empty((2, 0), dtype=np.int64))
    ramp_map: dict[str, tuple[int, int]] = field(default_factory=dict)

    @classmethod
    def load(cls, release: Path, panel: str) -> "PanelInfo":
        panel_dir = release / "corridors" / panel
        candidates = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
        if not candidates:
            raise FileNotFoundError(f"No original train observations under {panel_dir}")
        observed = pd.read_parquet(candidates[0], columns=["link_id"]).link_id.astype(str).unique().tolist()
        network = panel_dir / "network"
        topology = pd.read_csv(network / "lwr_mainline_topology.csv", dtype=str)
        # Latent mainline connectors carry no detector value, but preserve the
        # road between observed links during the spatial scan.
        links = scan_layout(sorted(set(observed) | set(topology.link_id.dropna().astype(str))), topology)
        frame = pd.read_csv(network / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        fd_path = network / "fd_parameters.csv"
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str}) if fd_path.exists() else pd.DataFrame()
        grouped = fd.groupby("link_id").mean(numeric_only=True) if not fd.empty else pd.DataFrame()

        def parameter(column: str, default: float) -> np.ndarray:
            raw = pd.to_numeric(frame[column], errors="coerce").reindex(links) if column in frame else pd.Series(default, index=links)
            if column in grouped:
                raw = pd.to_numeric(grouped[column], errors="coerce").reindex(links).fillna(raw)
            return raw.fillna(default).to_numpy(dtype=np.float32)

        free_speed = parameter("free_speed_kmh", 105.0)
        lanes = np.clip(parameter("lanes", 1.0), 1, 16)
        lengths = parameter("length_km", 1.0)
        station_lanes: dict[str, float] = {}
        if "station_id" in fd and "lanes" in fd:
            for row in fd.dropna(subset=["station_id", "lanes"]).itertuples(index=False):
                station_lanes[str(row.station_id)] = float(row.lanes)
        lookup = {link: i for i, link in enumerate(links)}
        pairs = set()
        for row in topology.itertuples(index=False):
            target = str(row.link_id)
            for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
                if source in lookup and target in lookup and source != target:
                    pairs.add((lookup[source], lookup[target]))
            for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
                if target in lookup and dest in lookup and dest != target:
                    pairs.add((lookup[target], lookup[dest]))
        ramp_map = {}
        ramp_path = network / "ramp_attachment_map.csv"
        if ramp_path.exists():
            for row in pd.read_csv(ramp_path, dtype=str).itertuples(index=False):
                if row.nearest_mainline_link_id in lookup:
                    if row.ramp_type not in {"OR", "FR", "on", "off"}:
                        raise ValueError(f"Unknown ramp type: {row.ramp_type}")
                    ramp_map[row.ramp_link_id] = (lookup[row.nearest_mainline_link_id], int(row.ramp_type in {"FR", "off"}))
        return cls(panel, links, lookup, free_speed, lanes, lengths, station_lanes,
                   parameter("capacity_vph", 6000.0), np.maximum(parameter("k_jam", np.nan), 1.05 * parameter("capacity_vph", 6000.0) / np.maximum(free_speed, 1)),
                   np.array(sorted(pairs), dtype=np.int64).reshape(-1, 2).T, ramp_map)

@dataclass
class StationDay:
    timestamps: list[str]
    station_ids: list[str]
    station_links: list[str]
    station_to_link: np.ndarray
    lanes: np.ndarray
    speed: np.ndarray
    flow_lane: np.ndarray
    eligible: np.ndarray
    occupancy: np.ndarray | None = None
    quality: np.ndarray | None = None

    @classmethod
    def load(cls, path: Path, panel: PanelInfo) -> "StationDay":
        columns = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "is_score_eligible"]
        # Optional published features also exist in Task 2 histories.
        import pyarrow.parquet as pq
        available = set(pq.read_schema(path).names) if path.stat().st_size else set(columns)
        columns += [c for c in ("occupancy", "pct_observed") if c in available]
        frame = pd.read_parquet(path, columns=columns)
        for column in ("timestamp", "station_id", "link_id"):
            frame[column] = frame[column].astype(str)
        if frame.duplicated(["timestamp", "station_id", "link_id"]).any():
            raise ValueError(f"Duplicate station/time rows in {path}")
        stations = frame[["station_id", "link_id"]].drop_duplicates()
        if stations.station_id.duplicated().any():
            raise ValueError(f"Station maps to multiple links in {path}")
        stations = stations.sort_values(["link_id", "station_id"])
        station_ids = stations.station_id.tolist()
        station_links = stations.link_id.tolist()
        unknown = set(station_links) - set(panel.link_index)
        if unknown:
            raise ValueError(f"Links absent from panel layout: {sorted(unknown)[:4]}")
        timestamps = sorted(frame.timestamp.unique().tolist())
        index = pd.MultiIndex.from_product([timestamps, station_ids], names=["timestamp", "station_id"])
        aligned = frame.set_index(["timestamp", "station_id"]).reindex(index)
        speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        lanes = np.array([panel.station_lanes.get(s, panel.lanes[panel.link_index[l]]) for s, l in zip(station_ids, station_links)], dtype=np.float32)
        lanes = np.clip(lanes, 1, 16)
        eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy(copy=True).reshape(len(timestamps), -1)
        eligible &= np.isfinite(speed) & np.isfinite(flow)
        occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(speed.shape) if "occupancy" in aligned else np.full_like(speed, np.nan)
        quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(speed.shape) / 100 if "pct_observed" in aligned else eligible.astype(np.float32)
        return cls(timestamps, station_ids, station_links,
                   np.array([panel.link_index[l] for l in station_links], dtype=np.int64),
                   lanes, speed / 25.0, flow / lanes[None, :] / 600.0, eligible, occupancy, quality)

def aggregate_channels(day: StationDay, panel: PanelInfo, visible: np.ndarray | None = None):
    values = np.stack((day.speed, day.flow_lane,
                       day.occupancy if day.occupancy is not None else np.full_like(day.speed, np.nan)), -1)
    good = np.isfinite(values)
    if visible is not None:
        good &= visible[..., None] if visible.ndim == 2 else visible
    sums = np.zeros((len(day.timestamps), len(panel.links), 3), np.float32)
    counts = np.zeros_like(sums)
    quality = np.zeros(sums.shape[:2], np.float32)
    qcount = np.zeros_like(quality)
    for station, link in enumerate(day.station_to_link):
        sums[:, link] += np.where(good[:, station], values[:, station], 0)
        counts[:, link] += good[:, station]
        present = good[:, station].any(-1)
        q = day.quality[:, station] if day.quality is not None else present.astype(np.float32)
        # Observation quality remains public even when measured values are masked.
        quality[:, link] += np.where(np.isfinite(q), q, 0)
        qcount[:, link] += np.isfinite(q)
    return sums / np.maximum(counts, 1), counts > 0, quality / np.maximum(qcount, 1)

def ramp_features(release: Path, panel: PanelInfo, split: str, timestamps: list[str]) -> np.ndarray:
    out = np.zeros((len(timestamps), len(panel.links), 4), np.float32)
    out[..., 2:] = 1  # A road without an attached ramp has known zero ramp flow.
    expected = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        expected[link, kind] += 1
    counts = np.zeros_like(out[..., :2])
    date = timestamps[0][:10]
    root = release / "corridors" / panel.name / split / "ramp_states"
    paths = sorted(set(root.glob(f"**/*{date}*.parquet")) | set(root.glob(f"**/*{date.replace('-', '_')}*.parquet")))
    time_index = {value: i for i, value in enumerate(timestamps)}
    for path in paths:
        frame = pd.read_parquet(path, columns=["timestamp", "ramp_link_id", "flow_vph", "pct_observed", "is_score_eligible"])
        frame = frame[frame.timestamp.isin(time_index)].drop_duplicates(["timestamp", "ramp_link_id"])
        for row in frame.itertuples(index=False):
            if row.ramp_link_id in panel.ramp_map and np.isfinite(row.flow_vph) and row.pct_observed >= 75 and bool(row.is_score_eligible):
                link, kind = panel.ramp_map[row.ramp_link_id]
                index = time_index[row.timestamp]
                out[index, link, kind] += float(row.flow_vph)
                counts[index, link, kind] += 1
    out[..., 2:] = counts >= expected[None]
    out[..., :2] *= out[..., 2:]
    return out

def dated_files(root):
    result = {}
    for path in sorted(Path(root).rglob("*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        if match is None:
            raise ValueError(f"Date missing from filename: {path}")
        date = match.group().replace("_", "-")
        if date in result:
            raise ValueError(f"Duplicate date: {date}")
        result[date] = path
    if not result:
        raise FileNotFoundError(f"No parquet files: {root}")
    return result

def fit_prior(panel, paths):
    """Only fit dates contribute to weekday/slot means and fallbacks."""
    sums = np.zeros((2016, len(panel.links), 3), np.float64)
    counts = np.zeros_like(sums)
    for number, path in enumerate(paths, 1):
        if number == 1 or number % 30 == 0 or number == len(paths):
            print(f"과거 평균 계산 {panel.name}: {number}/{len(paths)} days", flush=True)
        day = StationDay.load(path, panel)
        values, valid, _ = aggregate_channels(day, panel, day.eligible)
        dates = pd.to_datetime(day.timestamps, utc=True)
        slot = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
        np.add.at(sums, slot, values * valid)
        np.add.at(counts, slot, valid)
    mean = sums.sum((0, 1)) / np.maximum(counts.sum((0, 1)), 1)
    link_mean = np.divide(sums.sum(0), counts.sum(0),
                          out=np.broadcast_to(mean, sums.shape[1:]).copy(),
                          where=counts.sum(0) > 0)
    return np.divide(sums, counts, out=np.broadcast_to(link_mean, sums.shape).copy(),
                     where=counts > 0).astype(np.float32)

class TargetIndex:
    def __init__(self, release, name, day, split='train'):
        path = release / 'task1' / name / split / 'sample_submission_state.csv'
        self.stations, self.links = day.station_ids, day.station_links
        self.archive, self.regimes = {}, {}
        station_lookup = {s: i for i, s in enumerate(self.stations)}
        parts = {}
        for frame in pd.read_csv(path, usecols=KEYS, dtype=str, chunksize=200_000):
            positions = frame.station_id.map(station_lookup)
            if positions.isna().any() or set(frame.panel) != {name}:
                raise ValueError(f'템플릿의 panel/station이 잘못되었습니다: {path}')
            if not np.array_equal(np.asarray(self.links)[positions.to_numpy(int)], frame.link_id.to_numpy()):
                raise ValueError(f'템플릿 station/link 연결이 다릅니다: {path}')
            times = pd.to_datetime(frame.timestamp, utc=True, format='ISO8601')
            if ((times.dt.minute % 5 != 0) | (times.dt.second != 0)).any():
                raise ValueError('템플릿은 5분 시각이어야 합니다.')
            slots = times.dt.hour * 12 + times.dt.minute // 5
            frame = frame.assign(_date=frame.timestamp.str[:10],
                                 _cell=(slots * len(self.stations) + positions).astype(np.uint32))
            for date, part in frame.groupby('_date', sort=False):
                regimes = part.mask_regime.unique()
                if len(regimes) != 1 or regimes[0] not in REGIMES:
                    raise ValueError(f'날짜별 regime이 잘못되었습니다: {date}')
                if date in self.regimes and self.regimes[date] != regimes[0]:
                    raise ValueError(f'날짜별 regime이 바뀝니다: {date}')
                self.regimes[date] = regimes[0]
                parts.setdefault(date, []).append(part._cell.to_numpy())
        if not parts:
            raise ValueError(f'빈 템플릿: {path}')
        self.archive = {d: np.concatenate(p) for d, p in parts.items()}
        if any(len(np.unique(a)) != len(a) for a in self.archive.values()):
            raise ValueError(f'중복 대상 셀: {path}')

    def get(self, day, require_eligible=True):
        date = day.timestamps[0][:10]
        if (day.station_ids, day.station_links) != (self.stations, self.links) or date not in self.archive:
            raise ValueError(f'대상 관측소 구성이 달라졌습니다: {date}')
        if any(t[:10] != date for t in day.timestamps):
            raise ValueError('하루 파일에 여러 날짜가 있습니다.')
        slots = {int(t[11:13]) * 12 + int(t[14:16]) // 5: i for i, t in enumerate(day.timestamps)}
        mask = np.zeros(day.speed.shape, bool)
        cells = self.archive[date]
        mask[[slots[int(s)] for s in cells // len(self.stations)], cells % len(self.stations)] = True
        if require_eligible and (mask & ~day.eligible).any():
            raise ValueError(f'정답이 없는 대상 셀: {date}')
        return self.regimes[date], mask

def available_panels(release):
    release = Path(release).expanduser().resolve()
    if not (release / 'corridors').is_dir() or not (release / 'task1').is_dir():
        raise FileNotFoundError(f'RELEASE_ROOT에 corridors/와 task1/이 필요합니다: {release}')
    available = sorted(p.name for p in (release / 'corridors').iterdir() if p.is_dir())
    names = available if PANELS is None else sorted(set(PANELS))
    if not names or set(names) - set(available):
        raise ValueError(f'잘못된 PANELS: {names}')
    return names

def station_graph(panel, day):
    # 무관측 connector를 따라 실제 도로에서 가장 가까운 관측 링크를 찾습니다.
    observed = set(day.station_to_link.tolist())
    neighbours = [set() for _ in panel.links]
    for a, b in panel.edges.T:
        neighbours[a].add(int(b))
        neighbours[b].add(int(a))
    links = np.eye(len(panel.links), dtype=bool)
    for origin in observed:
        pending, seen = list(neighbours[origin]), {origin}
        while pending:
            node = pending.pop()
            if node in seen:
                continue
            seen.add(node)
            if node in observed:
                links[origin, node] = True
            else:
                pending.extend(neighbours[node] - seen)
    return links[day.station_to_link[:, None], day.station_to_link[None, :]]

def prepare_panel(release, name, code_id, station_offset):
    started = time.perf_counter()
    panel = PanelInfo.load(release, name)
    files = dated_files(release / 'corridors' / name / 'train' / 'mainline_states')
    masked = dated_files(release / 'corridors' / name / 'train' / 'mainline_states_masked')
    dates = {p: [d for d in files if lo <= d <= hi] for p, (lo, hi) in PERIODS.items()}
    if MAX_TRAIN_DAYS is not None:
        if MAX_TRAIN_DAYS < 1:
            raise ValueError('MAX_TRAIN_DAYS는 양수 또는 None이어야 합니다.')
        indices = np.linspace(0, len(dates['fit']) - 1, min(MAX_TRAIN_DAYS, len(dates['fit'])), dtype=int)
        dates['fit'] = [dates['fit'][i] for i in indices]
    if any(not dates[p] for p in ('fit', 'early', 'blend')):
        raise ValueError(f'학습·조기 종료·선택 구간이 모두 필요합니다: {name}')
    first = StationDay.load(files[dates['fit'][0]], panel)
    observed_links = np.unique(first.station_to_link)
    for attribute, fallback in (('free_speed', 100.), ('capacity', 1800. * panel.lanes), ('lengths', 1.)):
        values = getattr(panel, attribute)
        invalid = ~np.isfinite(values) | ((values < 0) if attribute == 'lengths' else (values <= 0))
        if invalid[observed_links].any():
            raise ValueError(f'관측 링크의 {attribute}가 잘못되었습니다: {name}')
        # 무관측 connector의 placeholder 값이 손실 연산에 NaN을 전파하지 않도록 합니다.
        setattr(panel, attribute, np.where(invalid, fallback, values).astype(np.float32))
    panel.jam_density = np.where(np.isfinite(panel.jam_density), panel.jam_density, 100 * panel.lanes)
    panel.jam_density = np.maximum(panel.jam_density, 1.05 * panel.capacity / panel.free_speed)
    target_index = TargetIndex(release, name, first)
    for p in ('fit', 'early', 'blend'):
        if {target_index.regimes.get(d) for d in dates[p]} != set(REGIMES):
            raise ValueError(f'{name}/{p}의 R1·R2·R3가 부족합니다. 날짜 수를 늘려 주세요.')
        if not set(dates[p]) <= set(masked):
            raise ValueError(f'masked 날짜가 누락되었습니다: {name}/{p}')
    mode = LWR_MODES.get(name, 'A').upper()
    if mode not in ('A', 'B', 'C'):
        raise ValueError(f'지원하지 않는 LWR mode: {mode}')
    out = dict(release=release, panel=panel, files=files, masked=masked, dates=dates,
               targets=target_index, profile=load_fit_profile(panel, target_index, dates['fit'], files),
               panel_code=code_id, station_codes=np.arange(station_offset, station_offset + len(first.station_ids)),
               mapping=first.station_to_link, lanes=first.lanes, station_graph=station_graph(panel, first),
               mode=mode)
    topology = pd.read_csv(release / 'corridors' / name / 'network' / 'lwr_mainline_topology.csv', dtype=str)
    out['topology_nodes'] = {panel.link_index[l] for l in topology.link_id.dropna().astype(str)}
    out['preparation_seconds'] = time.perf_counter() - started
    print({'panel': name, 'stations': len(first.station_ids), 'days': {p: len(ds) for p, ds in dates.items()},
           'preparation_seconds': round(out['preparation_seconds'], 1)}, flush=True)
    return out

def link_mean_numpy(values, active, mapping, nodes):
    sums = np.zeros((len(values), nodes, values.shape[-1]), np.float64)
    counts = np.zeros((len(values), nodes), np.float64)
    np.add.at(sums.transpose(1, 0, 2), mapping, np.where(active[..., None], values, 0).transpose(1, 0, 2))
    np.add.at(counts.T, mapping, active.T)
    return sums / np.maximum(counts[..., None], 1), counts

def nearest_features(values, prior, elapsed):
    good = np.isfinite(values)
    t = np.arange(len(values))[:, None]
    left = np.maximum.accumulate(np.where(good, t, -1), axis=0)
    right = np.minimum.accumulate(np.where(good, t, len(values))[::-1], axis=0)[::-1]
    lv, rv = left >= 0, right < len(values)
    ls, rs = np.clip(left, 0, len(values) - 1), np.clip(right, 0, len(values) - 1)
    a, b = np.take_along_axis(values, ls, axis=0), np.take_along_axis(values, rs, axis=0)
    a = np.where(lv, a, np.where(rv, b, prior))
    b = np.where(rv, b, a)
    la, ra = elapsed[:, None] - elapsed[ls], elapsed[rs] - elapsed[:, None]
    fraction = np.divide(la, la + ra, out=np.zeros_like(la), where=(la + ra) > 0).clip(0, 1)
    base = a + fraction * (b - a)
    la, ra = np.where(lv, la, np.nan), np.where(rv, ra, np.nan)
    return base, la, ra

def macro_mean(panel_values):
    families = {}
    for name, value in panel_values.items():
        families.setdefault(name.rsplit('_', 1)[0], []).append(value)
    return float(np.mean([np.mean(values) for values in families.values()]))


In [ ]:
AUX_NAMES = ('quality', 'speed_visible', 'flow_visible', 'occupancy_visible',
             'speed_left_age', 'speed_right_age', 'flow_left_age', 'flow_right_age',
             'profile_speed', 'profile_flow', 'profile_occupancy',
             'lanes', 'free_speed', 'capacity_per_lane', 'length',
             'ramp_in', 'ramp_out', 'ramp_in_valid', 'ramp_out_valid', 'on_attached', 'off_attached')


def restore_input_cache(names, archive_path, destination):
    destination = Path(destination).expanduser()
    destination.mkdir(parents=True, exist_ok=True)
    if archive_path is not None:
        archive_path = Path(archive_path).expanduser()
        if not archive_path.is_file():
            raise FileNotFoundError(f'저장 캐시 ZIP이 없습니다: {archive_path}. 새 캐시를 만들 때는 SAVED_CACHE_ZIP=None, CACHE_MODE="build"로 설정해 주세요.')
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            for member in members:
                relative = PurePosixPath(member.filename)
                if (relative.is_absolute() or '..' in relative.parts or '\\' in member.filename
                        or (member.external_attr >> 16) & 0o170000 == 0o120000):
                    raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
            fingerprint = hashlib.sha256(json.dumps(
                [(m.filename, m.file_size, m.CRC) for m in members], sort_keys=True).encode()).hexdigest()
            marker = destination / '.cache_source.json'
            if marker.exists() and json.loads(marker.read_text())['fingerprint'] != fingerprint:
                raise ValueError('다른 ZIP을 푼 폴더입니다. CACHE_DIR를 새 로컬 경로로 지정해 주세요.')
            pending = [m for m in members if not m.is_dir() and
                       (not marker.exists() or not (destination / m.filename).is_file())]
            if pending:
                needed = sum(m.file_size for m in pending)
                if shutil.disk_usage(destination).free < needed + 2 * 2**30:
                    raise RuntimeError(f'캐시 ZIP 복원에 {needed / 2**30:.1f} GiB + 여유 2 GiB가 필요합니다.')
                began = time.perf_counter()
                for number, member in enumerate(pending, 1):
                    target = destination / member.filename
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix(target.suffix + '.tmp')
                    with archive.open(member) as src, temporary.open('wb') as dst:
                        shutil.copyfileobj(src, dst)
                    temporary.replace(target)
                    if number == 1 or number % 100 == 0 or number == len(pending):
                        print(f'저장 ZIP 복원: {number}/{len(pending)}파일', flush=True)
                marker_temporary = marker.with_suffix('.json.tmp')
                marker_temporary.write_text(json.dumps(dict(fingerprint=fingerprint, archive=str(archive_path))))
                marker_temporary.replace(marker)
                print({'cache_restore_seconds': round(time.perf_counter() - began, 1)})
            else:
                print('같은 ZIP이 이미 복원되어 로컬 NPZ를 재사용합니다.')
    roots = {p.parent.parent.parent for p in destination.rglob('*.npz')
             if p.parent.name in PERIODS and p.parent.parent.name in names
             and re.fullmatch(r'\d{4}-\d{2}-\d{2}\.npz', p.name)}
    if len(roots) > 1:
        raise ValueError('ZIP에 캐시 루트가 여러 개입니다. 필요한 폴더를 CACHE_DIR로 지정하고 SAVED_CACHE_ZIP=None으로 설정해 주세요.')
    if not roots and CACHE_MODE == 'reuse':
        raise ValueError('캐시의 <패널>/fit|early|blend/YYYY-MM-DD.npz 구조를 찾지 못했습니다. 다른 모델의 NPZ는 사용할 수 없습니다.')
    return next(iter(roots)) if roots else destination, destination


def load_fit_profile(panel, targets, fit_dates, files):
    folder = CACHE_ROOT / panel.name
    folder.mkdir(parents=True, exist_ok=True)
    profile_path, metadata_path = folder / 'fit_profile.npy', folder / 'cache_metadata.json'
    metadata = dict(format='maginet-input-v1', panel=panel.name, links=panel.links,
                    stations=targets.stations, station_links=targets.links,
                    fit_dates=fit_dates, aux_names=list(AUX_NAMES), scale=SCALE.tolist())
    if metadata_path.exists():
        if json.loads(metadata_path.read_text()) != metadata:
            raise ValueError(f'캐시의 학습 날짜·관측소·입력 형식이 다릅니다: {panel.name}. CACHE_DIR를 다른 폴더로 지정해 주세요.')
        if not profile_path.is_file():
            raise FileNotFoundError(f'캐시 과거 평균이 누락되었습니다: {profile_path}')
        profile = np.load(profile_path, allow_pickle=False)
    else:
        candidates = [CACHE_PROFILE_DIR / f'fit_profile-{panel.name}.npy']
        candidates += sorted(CACHE_CONTAINER.rglob(f'fit_profile-{panel.name}.npy'))
        profile = None
        for candidate in dict.fromkeys(candidates):
            config_path = candidate.parent / 'config.json'
            if not candidate.is_file() or not config_path.is_file():
                continue
            config = json.loads(config_path.read_text())
            saved = config.get('panels', {}).get(panel.name, {})
            if (saved.get('stations') != targets.stations or saved.get('links') != targets.links
                    or saved.get('dates', {}).get('fit') != fit_dates):
                raise ValueError(f'이전 과거 평균의 관측소·fit 날짜가 다릅니다: {config_path}')
            profile = np.load(candidate, allow_pickle=False)
            if hashlib.sha256(profile.tobytes()).hexdigest() != saved.get('profile_hash'):
                raise ValueError(f'과거 평균의 hash가 설정과 다릅니다: {candidate}')
            print(f'저장 과거 평균 재사용: {candidate}', flush=True)
            break
        if profile is None:
            if CACHE_MODE == 'reuse':
                raise FileNotFoundError(f'{panel.name}: NPZ만으로 과거 평균을 복원할 수 없습니다. CACHE_PROFILE_DIR에 이전 full/config.json과 fit_profile-{panel.name}.npy가 필요합니다.')
            profile = fit_prior(panel, [files[d] for d in fit_dates])
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'저장 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        temporary = profile_path.with_suffix('.npy.tmp')
        with temporary.open('wb') as stream:
            np.save(stream, profile, allow_pickle=False)
        temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
    if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
        raise ValueError(f'캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
    return profile.astype(np.float32, copy=False)


def validate_cache_targets(data, context, date):
    required = {'values', 'visible', 'aux', 'slot', 'weekday', 'base', 'truth', 'target',
                'active', 'observed', 'timestamps', 'ti', 'si', 'regime', 'ramp_valid', 'consecutive'}
    if required - set(data):
        raise ValueError(f'이 MagiNet 노트북의 NPZ 형식이 아닙니다: {date}, 누락 {sorted(required - set(data))}')
    target, timestamps = data['target'], data['timestamps']
    stations = len(context['targets'].stations)
    if (target.dtype != np.bool_ or target.ndim != 2 or target.shape != (len(timestamps), stations)
            or len(set(timestamps.tolist())) != len(timestamps)
            or any(str(t)[:10] != date for t in timestamps)):
        raise ValueError(f'캐시 날짜·관측소 형태가 다릅니다: {date}')
    ti, si = np.nonzero(target)
    cells = np.array([int(str(t)[11:13]) * 12 + int(str(t)[14:16]) // 5
                      for t in timestamps], np.int64)[ti] * stations + si
    expected = context['targets'].archive[date]
    if (not np.array_equal(np.sort(cells), np.sort(expected))
            or not np.array_equal(ti, data['ti']) or not np.array_equal(si, data['si'])
            or int(data['regime']) != REGIMES.index(context['targets'].regimes[date])):
        raise ValueError(f'캐시 대상 셀·regime이 템플릿과 다릅니다: {date}')
    return int(data['regime']), len(ti)


def fit_input_normalization(entries, cache_folder, mode=INPUT_NORMALIZATION):
    fit = sorted((e for e in entries if e['period'] == 'fit'), key=lambda e: (e['panel'], e['date']))
    if not fit or mode not in ('fixed', 'standard'):
        raise ValueError('표준화에는 fit 캐시와 fixed/standard 모드가 필요합니다.')
    provenance = []
    for entry in fit:
        # NPZ의 배열 CRC는 ZIP 복원·경로 이동·수정 시각 변경 후에도 같습니다.
        with zipfile.ZipFile(entry['path']) as archive:
            arrays = [(name, archive.getinfo(name).file_size, archive.getinfo(name).CRC)
                      for name in ('values.npy', 'visible.npy')]
        provenance.append((entry['panel'], entry['date'], arrays))
    key = hashlib.sha256(json.dumps(dict(format='fit-visible-crc-v1', mode=mode, files=provenance),
                                   sort_keys=True).encode()).hexdigest()
    path = Path(cache_folder) / f'input_normalization-{key[:16]}.json'
    if path.exists():
        saved = json.loads(path.read_text())
        if saved.get('source_signature') != key or saved.get('mode') != mode:
            raise ValueError('저장된 입력 표준화 설정이 다릅니다.')
        return saved
    count, total, square = np.zeros(3, np.int64), np.zeros(3, np.float64), np.zeros(3, np.float64)
    if mode == 'standard':
        for number, entry in enumerate(fit, 1):
            with np.load(entry['path'], allow_pickle=False) as data:
                values, visible = data['values'], data['visible']
                if (values.shape != visible.shape or visible.dtype != np.bool_
                        or values.ndim != 3 or values.shape[-1] != 3):
                    raise ValueError(f'표준화 입력 형태가 다릅니다: {entry["path"]}')
                if not np.isfinite(values[visible]).all():
                    raise ValueError(f'관측된 입력에 NaN/Inf가 있습니다: {entry["path"]}')
                observed = np.where(visible, values, 0).astype(np.float64)
                count += visible.sum((0, 1))
                total += observed.sum((0, 1))
                square += np.square(observed).sum((0, 1))
            if number == 1 or number % 100 == 0 or number == len(fit):
                print(f'fit 관측 입력 표준화: {number}/{len(fit)}일', flush=True)
    mean = total / np.maximum(count, 1)
    std = np.sqrt(np.maximum(square / np.maximum(count, 1) - mean**2, 0))
    std = np.where(std > 1e-6, std, 1.)
    saved = dict(mode=mode, mean=mean.tolist(), std=std.tolist(), count=count.tolist(),
                 source_signature=key, fitted_period='fit', channels=['speed/25', 'flow_per_lane/600', 'occupancy'])
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(saved, indent=2, ensure_ascii=False))
    temporary.replace(path)
    return saved


def attach_vocabulary(contexts):
    families = sorted({n.rsplit('_', 1)[0] for n in contexts})
    for name, c in contexts.items():
        family, direction = name.rsplit('_', 1)
        if direction not in ('N', 'S', 'E', 'W'):
            raise ValueError(f'알 수 없는 방향: {name}')
        c['family_code'] = families.index(family)
        c['direction_code'] = ('N', 'S', 'E', 'W').index(direction)


def chebyshev_supports(adjacency, order=CHEB_K):
    adjacency = np.asarray(adjacency, np.float64)
    if adjacency.ndim != 2 or adjacency.shape[0] != adjacency.shape[1] or not np.allclose(adjacency, adjacency.T):
        raise ValueError('Chebyshev 그래프는 대칭 인접행렬이어야 합니다.')
    degree = adjacency.sum(-1)
    inv = 1 / np.sqrt(np.maximum(degree, 1))
    identity = np.eye(len(adjacency))
    laplacian = identity - inv[:, None] * adjacency * inv[None, :]
    largest = float(np.linalg.eigvalsh(laplacian).max())
    scaled = 2 * laplacian / largest - identity if largest > 1e-10 else -identity
    supports = [identity]
    if order > 1:
        supports.append(scaled)
    for _ in range(2, order):
        supports.append(2 * scaled @ supports[-1] - supports[-2])
    return np.stack(supports).astype(np.float32)


def input_arrays(day, context, split='train'):
    panel, mapping = context['panel'], day.station_to_link
    values = np.stack((day.speed, day.flow_lane, day.occupancy), -1)
    visible = np.isfinite(values)
    dates = pd.to_datetime(day.timestamps, utc=True, format='ISO8601')
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 60, np.float32)
    if len(elapsed) < 2 or (np.diff(elapsed) <= 0).any():
        raise ValueError('하루 시각은 서로 다른 오름차순이어야 합니다.')
    slot, weekday = np.asarray(dates.hour * 12 + dates.minute // 5), np.asarray(dates.dayofweek)
    prior = context['profile'][(weekday * 288 + slot)[:, None], mapping[None, :]]
    baseline, ages = [], []
    for channel, data in enumerate((day.speed, day.flow_lane)):
        base, left_age, right_age = nearest_features(data, prior[..., channel], elapsed)
        baseline.append(base)
        ages += [np.nan_to_num(age / 360, nan=4.).clip(0, 4) for age in (left_age, right_age)]
    ramps = ramp_features(context['release'], panel, split, day.timestamps)
    r = ramps[:, mapping]
    attached = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        attached[link, kind] = 1
    spread = lambda v: np.broadcast_to(v, day.speed.shape)
    aux = np.stack((day.quality, visible[..., 0], visible[..., 1], visible[..., 2], *ages,
                    prior[..., 0], prior[..., 1], prior[..., 2], spread(day.lanes / 5),
                    spread(panel.free_speed[mapping] / 25), spread(panel.capacity[mapping] / panel.lanes[mapping] / 600),
                    spread(panel.lengths[mapping]), r[..., 0] / 600, r[..., 1] / 600, r[..., 2], r[..., 3],
                    spread(attached[mapping, 0]), spread(attached[mapping, 1])), -1)
    if aux.shape[-1] != len(AUX_NAMES) or np.isinf(values).any():
        raise ValueError('입력 배열이 잘못되었습니다.')
    # 수치상 placeholder 0은 mask로 제외되며 관측값 임베딩으로 사용되지 않습니다.
    values = np.where(visible, values, 0).astype(np.float32)
    aux = np.nan_to_num(aux, nan=0., posinf=10., neginf=-10.).clip(-10, 10).astype(np.float32)
    base = np.stack(baseline, -1).astype(np.float32)
    if not np.isfinite(base).all():
        raise ValueError('기준 보간이 유한하지 않습니다.')
    return dict(values=values, visible=visible, aux=aux, slot=slot.astype(np.int64),
                weekday=weekday.astype(np.int64), base=base,
                ramp_valid=(ramps[..., 2] > 0) & (ramps[..., 3] > 0))


def consecutive_five_minutes(timestamps):
    times = pd.to_datetime(timestamps, utc=True, format='ISO8601')
    return np.asarray((times[1:] - times[:-1]) == pd.Timedelta(minutes=5))


def make_day(context, date, split='train', target_index=None, input_files=None):
    panel = context['panel']
    path = context['masked'][date] if split == 'train' else input_files[date]
    day = StationDay.load(path, panel)
    if split == 'train':
        truth = StationDay.load(context['files'][date], panel)
        if (day.timestamps, day.station_ids, day.station_links) != (truth.timestamps, truth.station_ids, truth.station_links):
            raise ValueError(f'정답과 masked 입력이 다릅니다: {date}')
        regime, target = context['targets'].get(truth)
    else:
        truth = None
        regime, target = target_index.get(day, require_eligible=False)
    if (day.station_ids, day.station_links) != (context['targets'].stations, context['targets'].links):
        raise ValueError(f'관측소 구성이 다릅니다: {date}')
    path_regimes = [p.name.split('=', 1)[1] for p in path.parents if p.name.startswith('mask_regime=')]
    if path_regimes and path_regimes[0] != regime:
        raise ValueError(f'입력과 템플릿의 regime이 다릅니다: {date}')
    if np.isfinite(day.speed[target]).any() or np.isfinite(day.flow_lane[target]).any():
        raise ValueError(f'대상 정답이 입력에 보입니다: {date}')
    d = input_arrays(day, context, split)
    observed_values = np.stack((day.speed * 25, day.flow_lane * 600 * day.lanes), -1)
    observed = np.isfinite(observed_values).all(-1)
    active = (observed & (truth.eligible if truth is not None else day.eligible)) | target
    labels = np.stack((truth.speed, truth.flow_lane), -1) if truth is not None else np.zeros((*day.speed.shape, 2), np.float32)
    ti, si = np.nonzero(target)
    d.update(truth=np.nan_to_num(labels), target=target, active=active,
             observed=np.nan_to_num(observed_values), timestamps=np.asarray(day.timestamps), ti=ti, si=si,
             regime=np.array(REGIMES.index(regime), np.int32),
             consecutive=consecutive_five_minutes(day.timestamps))
    return d


def cache_context(context, cache_root, date_overrides=None, periods=('fit', 'early', 'blend')):
    started, entries = time.perf_counter(), []
    for period in periods:
        dates = context['dates'][period] if date_overrides is None else date_overrides[period]
        folder = cache_root / context['panel'].name / period
        folder.mkdir(parents=True, exist_ok=True)
        for number, date in enumerate(dates, 1):
            path = folder / f'{date}.npz'
            if not path.exists():
                if CACHE_MODE == 'reuse':
                    raise FileNotFoundError(f'저장 캐시가 누락되었습니다: {path}. 원본에서 생성하려면 CACHE_MODE="build"로 변경해 주세요.')
                if shutil.disk_usage(cache_root).free < 2 * 2**30:
                    raise RuntimeError('임시 캐시 디스크 여유 공간이 2 GiB 미만입니다.')
                d = make_day(context, date)
                temporary = path.with_suffix('.npz.tmp')
                with temporary.open('wb') as stream:
                    np.savez_compressed(stream, **d)
                temporary.replace(path)
                regime, targets = validate_cache_targets(d, context, date)
            else:
                with np.load(path, allow_pickle=False) as d:
                    regime, targets = validate_cache_targets(d, context, date)
            entries.append(dict(panel=context['panel'].name, date=date, period=period, regime=regime,
                                targets=targets, path=str(path)))
            if number == 1 or number % 20 == 0 or number == len(dates):
                print(f"입력 캐시 {context['panel'].name}/{period}: {number}/{len(dates)}일", flush=True)
    context['cache_seconds'] = context.get('cache_seconds', 0.) + time.perf_counter() - started
    return entries


@lru_cache(maxsize=2)
def load_day(path):
    with np.load(path, allow_pickle=False) as d:
        result = {k: d[k] for k in d.files}
    values, visible, base, aux = (result[k] for k in ('values', 'visible', 'base', 'aux'))
    if (values.ndim != 3 or values.shape[-1] != 3 or visible.dtype != np.bool_
            or visible.shape != values.shape or base.shape != values.shape[:2] + (2,)
            or aux.shape != values.shape[:2] + (len(AUX_NAMES),)
            or not np.isfinite(values[visible]).all() or not np.isfinite(base).all()
            or not np.isfinite(aux).all() or (base < 0).any()
            or visible[result['target'], :2].any()):
        raise ValueError(f'캐시 배열 형태·관측 마스크·보간값이 잘못되었습니다: {path}')
    return result


def window_starts(length):
    if length <= WINDOW:
        return [0]
    return sorted(set(range(0, length - WINDOW + 1, STRIDE)) | {length - WINDOW})


## 3. MagiNet AMSTenc·MASTdec 구현

채널별 관측 임베딩과 학습 가능한 결측 토큰을 마스크로 선택합니다. MASTdec는 시간 Attention, 창 내 표현으로 만드는 공간 Attention과 missing graph, Chebyshev 합성곱, 3·5·7 시간 게이트를 쌓습니다.

노드 수에 고정된 학습 행렬은 노드 표현에서 만드는 행렬로 바꾸고 그래프 support는 패널별로 전달합니다. 모든 키가 결측이면 Attention 가중치를 0으로 처리합니다. 초기 인코더와 각 블록 출력을 결합하고, 시간축 양방향 GRU와 속도·교통량의 별도 head로 양·음 보정량을 읽고 보간값에 더합니다. 보정 head의 마지막 층은 0으로 초기화해 시작 출력이 보간값과 같습니다. fit 표준화의 평균·표준편차는 모델 버퍼로 저장하며 추론에서도 같은 기준을 적용합니다.

In [ ]:
def masked_softmax(scores, valid):
    valid = valid.to(dtype=torch.bool)
    weights = scores.float().masked_fill(~valid, torch.finfo(torch.float32).min).softmax(-1)
    weights = torch.where(valid, weights, 0.)
    return weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)


class AMSTEncoder(nn.Module):
    def __init__(self, station_count, panel_count, family_count):
        super().__init__()
        self.value_weight = nn.Parameter(torch.empty(3, WIDTH))
        self.value_bias = nn.Parameter(torch.zeros(3, WIDTH))
        self.missing_token = nn.Parameter(torch.empty(3, WIDTH))
        self.slot = nn.Embedding(288, WIDTH)
        self.weekday = nn.Embedding(7, WIDTH)
        self.relative = nn.Embedding(288, WIDTH)
        self.station = nn.Embedding(station_count, WIDTH)
        self.panel = nn.Embedding(panel_count, WIDTH)
        self.family = nn.Embedding(family_count, WIDTH)
        self.direction = nn.Embedding(4, WIDTH)
        self.regime = nn.Embedding(3, WIDTH)
        self.aux = nn.Linear(len(AUX_NAMES), WIDTH)
        self.base_embedding = nn.Linear(2, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.value_weight)
        nn.init.normal_(self.missing_token, std=0.02)

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        # values/visible: B×T×station×3; 결측 placeholder는 여기서 제외합니다.
        tokens = values[..., None] * self.value_weight + self.value_bias
        tokens = torch.where(visible[..., None], tokens, self.missing_token)
        h = tokens.sum(-2) / np.sqrt(3.)
        t = len(values[0])
        time_embedding = self.slot(slot) + self.weekday(weekday) + self.relative(torch.arange(t, device=values.device))[None]
        node_embedding = (self.station(ct['station_codes']) + self.panel(ct['panel_code'])
                          + self.family(ct['family_code']) + self.direction(ct['direction_code']))
        h = h + self.aux(aux) + self.base_embedding(base) + time_embedding[:, :, None] + node_embedding[None, None] + self.regime(regime)[:, None, None]
        return self.dropout(self.norm(h)).permute(0, 2, 1, 3), node_embedding


class MaskTemporalAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(WIDTH, 3 * WIDTH, bias=False)
        self.project = nn.Linear(WIDTH, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, residual_attention):
        b, s, t, d = h.shape
        q, k, v = self.qkv(h).reshape(b, s, t, 3, HEADS, d // HEADS).permute(3, 0, 1, 4, 2, 5).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / np.sqrt(d // HEADS)
        if residual_attention is not None:
            scores = scores + residual_attention
        key_valid = visible.any(-1).permute(0, 2, 1)[:, :, None, None, :]
        weights = masked_softmax(scores, key_valid)
        attended = (weights.to(v.dtype) @ v).transpose(2, 3).reshape(b, s, t, d)
        return self.norm(h + self.dropout(self.project(attended))), scores


class MaskSpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.summary = nn.Linear(2 * WIDTH, GRAPH_DIM)
        self.position = nn.Linear(WIDTH, GRAPH_DIM)
        self.qk = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 16, bias=False)
        self.mask_pair = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 8, bias=False)
        self.norm = nn.LayerNorm(GRAPH_DIM)

    def forward(self, h, visible, node_embedding):
        b, s, t, d = h.shape
        valid = visible.any(-1).permute(0, 2, 1).float()
        observed_summary = (h * valid[..., None]).sum(2) / valid.sum(2).clamp_min(1)[..., None]
        summary = self.summary(torch.cat((observed_summary, h.mean(2)), -1))
        # MagiNet의 관측 문맥 기반 missing spatial graph를 유지합니다.
        similarity = F.relu(summary.float() @ summary.float().transpose(-1, -2))
        missing_graph = (similarity.softmax(-1) > MGL_THRESHOLD).float()
        positioned = self.norm(summary + self.position(node_embedding)[None])
        q, k = self.qk(positioned).reshape(b, s, 2, CHEB_K, 16).permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / 4.
        a, z = self.mask_pair(positioned).reshape(b, s, 2, CHEB_K, 8).permute(2, 0, 3, 1, 4).unbind(0)
        pair_mask = torch.tanh((a.float() @ z.float().transpose(-1, -2)) / np.sqrt(8.))
        reliability = 0.25 + 0.75 * valid.mean(2)
        scores = scores + missing_graph[:, None] * pair_mask + reliability.log()[:, None, None, :]
        return scores.softmax(-1)


class MaskAwareChebConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.empty(CHEB_K, WIDTH, WIDTH))
        nn.init.xavier_uniform_(self.theta)

    def forward(self, h, spatial_attention, supports):
        weights = spatial_attention * supports[None]
        output = torch.einsum('bkij,bjtd,kdf->bitf', weights.to(h.dtype), h, self.theta.to(h.dtype))
        return F.relu(output)


class GatedTemporalUnit(nn.Module):
    def __init__(self, kernel):
        super().__init__()
        self.conv = nn.Conv2d(WIDTH, 2 * WIDTH, kernel_size=(1, kernel), padding=(0, kernel // 2))

    def forward(self, h):
        a, b = self.conv(h).chunk(2, dim=1)
        return a.tanh() * b.sigmoid()


class MASTBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal_attention = MaskTemporalAttention()
        self.spatial_attention = MaskSpatialAttention()
        self.spatial_conv = MaskAwareChebConv()
        self.gates = nn.ModuleList(GatedTemporalUnit(k) for k in (3, 5, 7))
        self.time_mix = nn.Conv2d(3 * WIDTH, WIDTH, kernel_size=1)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, node_embedding, supports, residual_attention):
        attended, residual_attention = self.temporal_attention(h, visible, residual_attention)
        spatial_attention = self.spatial_attention(attended, visible, node_embedding)
        spatial = self.spatial_conv(h, spatial_attention, supports)
        channels = spatial.permute(0, 3, 1, 2)
        gated = self.time_mix(torch.cat([gate(channels) for gate in self.gates], dim=1)).permute(0, 2, 3, 1)
        output = self.norm(h + self.dropout(F.relu(spatial + gated)))
        return output, residual_attention


class MagiNetTask13(nn.Module):
    def __init__(self, station_count, panel_count, family_count, normalization):
        super().__init__()
        mean = torch.as_tensor(normalization['mean'], dtype=torch.float32)
        std = torch.as_tensor(normalization['std'], dtype=torch.float32)
        if mean.shape != (3,) or std.shape != (3,) or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or (std <= 0).any():
            raise ValueError('입력 표준화의 평균·표준편차가 잘못되었습니다.')
        self.register_buffer('input_mean', mean)
        self.register_buffer('input_std', std)
        self.AMSTenc = AMSTEncoder(station_count, panel_count, family_count)
        self.MASTdec = nn.ModuleList(MASTBlock() for _ in range(BLOCKS))
        self.fuse = nn.Linear((BLOCKS + 1) * WIDTH, WIDTH)
        self.readout = nn.GRU(WIDTH, WIDTH // 2, batch_first=True, bidirectional=True)
        self.speed_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        self.flow_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        for head in (self.speed_head, self.flow_head):
            # 보정량은 양·음 모두 허용하며 시작 출력은 기존 보간과 같습니다.
            nn.init.zeros_(head[-1].weight)
            nn.init.zeros_(head[-1].bias)

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        normalized = torch.where(visible, (values.float() - self.input_mean) / self.input_std, 0.)
        normalized_base = (base.float() - self.input_mean[:2]) / self.input_std[:2]
        normalized_aux = aux.float().clone()
        normalized_aux[..., 8:11] = (normalized_aux[..., 8:11] - self.input_mean) / self.input_std
        h, node_embedding = self.AMSTenc(normalized, visible, normalized_aux, normalized_base, slot, weekday, regime, ct)
        outputs, residual_attention = [h], None
        for block in self.MASTdec:
            h, residual_attention = block(h, visible, node_embedding, ct['supports'], residual_attention)
            outputs.append(h)
        h = F.gelu(self.fuse(torch.cat(outputs, dim=-1)))
        b, s, t, d = h.shape
        # cuDNN의 BF16 RNN 지원 여부와 무관하게 시간 readout은 FP32로 계산합니다.
        with torch.autocast(device_type=h.device.type, enabled=False):
            h, _ = self.readout(h.float().reshape(b * s, t, d))
        h = h.reshape(b, s, t, d).permute(0, 2, 1, 3)
        correction = torch.cat((self.speed_head(h), self.flow_head(h)), -1).float()
        # 입력 표준화와 별개로 출력·State 손실은 /25·/600 단위를 유지합니다.
        return (base.float() + correction).clamp_min(0.)


def tensor_context(context, device=DEVICE):
    panel = context['panel']
    mapping = torch.as_tensor(context['mapping'], dtype=torch.long, device=device)
    return dict(station_codes=torch.as_tensor(context['station_codes'], dtype=torch.long, device=device),
                panel_code=torch.tensor(context['panel_code'], dtype=torch.long, device=device),
                family_code=torch.tensor(context['family_code'], dtype=torch.long, device=device),
                direction_code=torch.tensor(context['direction_code'], dtype=torch.long, device=device),
                supports=torch.as_tensor(chebyshev_supports(context['station_graph']), device=device),
                matrix=F.one_hot(mapping, num_classes=len(panel.links)).float(),
                lanes=torch.as_tensor(context['lanes'], device=device),
                link_lanes=torch.as_tensor(panel.lanes, device=device),
                lengths=torch.as_tensor(panel.lengths, device=device), free=torch.as_tensor(panel.free_speed, device=device),
                capacity=torch.as_tensor(panel.capacity, device=device), jam=torch.as_tensor(panel.jam_density, device=device))


def window_batch(d, starts, device=DEVICE):
    lengths = [min(WINDOW, len(d['values']) - start) for start in starts]
    if len(set(lengths)) != 1:
        raise ValueError('한 batch의 시간 길이가 다릅니다.')
    return {key: torch.as_tensor(np.stack([d[key][start:start+length] for start, length in zip(starts, lengths)]),
                                 device=device, dtype=dtype)
            for key, dtype in (('values', torch.float32), ('visible', torch.bool), ('aux', torch.float32),
                                ('base', torch.float32), ('slot', torch.long), ('weekday', torch.long))} | {
               'regime': torch.full((len(starts),), int(d['regime']), dtype=torch.long, device=device)}


@torch.inference_mode()
def maginet_predict(model, d, ct, batch_size=BATCH_SIZE):
    model.eval()
    device = next(model.parameters()).device
    sums = np.zeros_like(d['base'], np.float64)
    weights = np.zeros((len(d['base']), 1, 1), np.float64)
    starts = window_starts(len(d['values']))
    for offset in range(0, len(starts), batch_size):
        chunk = starts[offset:offset+batch_size]
        batch = window_batch(d, chunk, device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP and device.type == 'cuda'):
            outputs = model(**batch, ct=ct).float().cpu().numpy()
        for start, output in zip(chunk, outputs):
            w = np.hanning(len(output) + 2)[1:-1, None, None]
            sums[start:start+len(output)] += output * w
            weights[start:start+len(output)] += w
    if (weights == 0).any():
        raise ValueError('겹치는 창 예측에 누락 시각이 있습니다.')
    result = (sums / weights).astype(np.float32)
    if not np.isfinite(result).all() or (result < 0).any():
        raise FloatingPointError('MagiNet 출력이 잘못되었습니다.')
    return result


## 4. Task 1 손실과 Task 3 보조 손실

Task 1은 정확한 대상 셀의 정규화 속도·차로당 교통량 오차를 `0.54:0.46`으로 학습합니다. 공식 State 평가는 RMSE로 따로 계산합니다. 공개 관측값은 유지하고 대상 셀만 예측으로 바꾼 뒤 관측소 값을 링크별로 평균냅니다.

`N = (링크 총 교통량 / max(속도, 1)) × 길이`이며 5분 인접 시각의 `ΔN`도 정답과 비교합니다. 변화량 자체를 0으로 만드는 손실은 사용하지 않습니다. 초기 5 epoch는 State만 학습하고 다음 5 epoch에 N·ΔN 가중치를 늘립니다. FD 가중치는 기본 0입니다.

로컬 LWR 진단은 같은 공개 관측을 완성한 예측과 정답의 ΔN 차이를 비교합니다. 숨은 boundary flux, 무관측 connector, 날짜 사이 변화량까지 포함한 공식 LWR 점수와 다릅니다. 아래 선택식 `0.70 State + 0.10 FD_proxy + 0.20 LWR_proxy`도 대리 기준입니다. 관측 가능한 물리 평가 셀이 없는 그룹은 해당 대리 점수를 0으로 두고 개수를 보고합니다.

In [ ]:
def observability_mask(context, counts, ramp_valid, affected, consecutive):
    panel = context['panel']
    present = (counts > 0).any(0)
    topology_nodes = context['topology_nodes']
    observable = present & np.array([i in topology_nodes for i in range(len(panel.links))])
    for a, b in panel.edges.T:
        if a in topology_nodes and not present[a]:
            observable[b] = False
        if b in topology_nodes and not present[b]:
            observable[a] = False
    if context['mode'] == 'C':
        for link, _ in panel.ramp_map.values():
            observable[link] = False
    transition = (counts[:-1] > 0) & (counts[1:] > 0) & observable[None] & consecutive[:, None]
    if context['mode'] in ('A', 'B'):
        transition &= ramp_valid[:-1]
    return transition, (affected[:-1] | affected[1:])

def completed_link_fields(prediction_physical, d, context):
    values = d['observed'].copy()
    ti, si = d['ti'], d['si']
    values[ti, si, 0] = prediction_physical[:, 0]
    values[ti, si, 1] = prediction_physical[:, 1] * context['lanes'][si]
    links, counts = link_mean_numpy(values, d['active'], context['mapping'], len(context['panel'].links))
    affected = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                              context['mapping'], len(context['panel'].links))[0][..., 0] > 0
    return links, counts, affected

def physics_totals(prediction, d, context):
    panel = context['panel']
    links, counts, affected = completed_link_fields(prediction, d, context)
    truth = d['truth'][d['ti'], d['si']] * SCALE
    reference, _, _ = completed_link_fields(truth, d, context)
    n = links[..., 1] / np.maximum(links[..., 0], 1) * panel.lengths
    true_n = reference[..., 1] / np.maximum(reference[..., 0], 1) * panel.lengths
    good_n = (counts > 0) & affected
    error_n = (n - true_n)[good_n]
    transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
    error_dn_all = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition]
    truth_dn_all = np.diff(true_n, axis=0)[transition]
    error_dn_target = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition & target_transition]
    density = links[..., 1] / np.maximum(links[..., 0], 1)
    critical = panel.capacity / panel.free_speed
    wave = panel.capacity / np.maximum(panel.jam_density - critical, 1e-6)
    qfd = np.where(density <= critical, panel.free_speed * density,
                   wave * (panel.jam_density - density)).clip(0)
    good_fd = counts > 0
    fd_error = ((links[..., 1] - qfd) / panel.lanes)[good_fd]
    q_lane = (links[..., 1] / panel.lanes)[good_fd]
    fraction = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                               context['mapping'], len(panel.links))[0][..., 0]
    submitted = good_fd & (fraction > 0.5)
    return dict(n_sae=float(np.abs(error_n).sum()), n_sse=float(np.square(error_n).sum()), n_count=len(error_n),
                dn_sae=float(np.abs(error_dn_target).sum()), dn_sse=float(np.square(error_dn_target).sum()),
                dn_count=len(error_dn_target), lwr_num=float(np.abs(error_dn_all).sum()),
                lwr_den=float(np.abs(truth_dn_all).sum()), lwr_count=len(error_dn_all),
                fd_num=float(np.abs(fd_error).sum()), fd_den=float(np.abs(q_lane).sum()),
                fd_count=int(good_fd.sum()), submitted_count=int(submitted.sum()),
                low_flow=int((links[..., 1][submitted] < 50).sum()))

def state_totals(truth, prediction):
    error = np.asarray(prediction, np.float64) - truth
    return dict(count=len(error), speed_sse=float(np.square(error[:, 0]).sum()),
                flow_sse=float(np.square(error[:, 1]).sum()), speed_sae=float(np.abs(error[:, 0]).sum()),
                flow_sae=float(np.abs(error[:, 1]).sum()))


In [ ]:
def summarize(records):
    daily = pd.DataFrame(records)
    if daily.empty:
        raise ValueError('평가 대상이 없습니다.')
    keys = ['candidate', 'alpha', 'panel', 'regime']
    sums = daily.groupby(keys, as_index=False).sum(numeric_only=True)
    expected = {(p, r) for p in daily.panel.unique() for r in range(3)}
    for _, part in sums.groupby('candidate'):
        if set(zip(part.panel, part.regime)) != expected or (part['count'] == 0).any():
            raise ValueError('평가의 panel×regime이 누락되었습니다.')
    sums['speed_rmse_kmh'] = np.sqrt(sums.speed_sse / sums['count'])
    sums['flow_rmse_per_lane'] = np.sqrt(sums.flow_sse / sums['count'])
    sums['speed_mae_kmh'] = sums.speed_sae / sums['count']
    sums['flow_mae_per_lane'] = sums.flow_sae / sums['count']
    sums['State'] = 0.54 * np.maximum(0, 1 - sums.speed_rmse_kmh / 25) + 0.46 * np.maximum(0, 1 - sums.flow_rmse_per_lane / 600)
    if 'lwr_num' in sums:
        sums['N_MAE_proxy'] = sums.n_sae / sums.n_count.replace(0, np.nan)
        sums['DN_MAE_proxy'] = sums.dn_sae / sums.dn_count.replace(0, np.nan)
        sums['LWR_proxy'] = np.maximum(0, 1 - sums.lwr_num / (sums.lwr_den + 1e-6))
        sums['FD_proxy'] = np.maximum(0, 1 - sums.fd_num / (sums.fd_den + 1e-6))
        sums.loc[sums.lwr_count == 0, 'LWR_proxy'] = 0.
        sums.loc[sums.fd_count == 0, 'FD_proxy'] = 0.
        sums['low_flow_fraction'] = sums.low_flow / sums.submitted_count.replace(0, np.nan)
        sums.loc[sums.low_flow_fraction > 0.20, 'FD_proxy'] = 0.
        sums['Physics_proxy'] = (sums.FD_proxy + 2 * sums.LWR_proxy) / 3
    metrics = ['State', 'speed_rmse_kmh', 'flow_rmse_per_lane', 'speed_mae_kmh', 'flow_mae_per_lane']
    metrics += [n for n in ('N_MAE_proxy', 'DN_MAE_proxy', 'FD_proxy', 'LWR_proxy', 'Physics_proxy') if n in sums]
    rows = []
    for name, part in sums.groupby('candidate', sort=False):
        per_panel = part.groupby('panel')[metrics].mean()
        row = dict(candidate=name, alpha=float(part.alpha.iloc[0]),
                   **{m: macro_mean(per_panel[m].to_dict()) for m in metrics})
        if 'FD_proxy' in row:
            row['selection_proxy'] = sum(row[k] * w for k, w in SELECT_WEIGHTS.items()) / sum(SELECT_WEIGHTS.values())
        rows.append(row)
    return pd.DataFrame(rows), sums, daily


In [ ]:
def training_transitions(d, context):
    if '_transition' not in d:
        _, counts, affected = completed_link_fields(d['truth'][d['ti'], d['si']] * SCALE, d, context)
        transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
        d['_transition'] = transition & target_transition
    return d['_transition']


def joint_loss(output, d, starts, context, ct, physics_factor):
    device, length = output.device, output.shape[1]
    take = lambda key: torch.as_tensor(np.stack([d[key][s:s+length] for s in starts]), device=device)
    target, truth = take('target').bool(), take('truth').float()
    if not target.any():
        raise ValueError('학습 창에 대상 셀이 없습니다.')
    state_error = (output.float() - truth)[target]
    state = 0.54 * state_error[:, 0].square().mean() + 0.46 * state_error[:, 1].square().mean()
    zero = state * 0.
    losses = dict(State=state, N=zero, dN=zero, FD=zero)
    if physics_factor > 0:
        active, observed = take('active').float(), take('observed').float()
        units = torch.stack((torch.full_like(ct['lanes'], 25.), 600 * ct['lanes']), -1)
        predicted = torch.where(target[..., None], output.float() * units, observed)
        reference = torch.where(target[..., None], truth * units, observed)
        counts = torch.einsum('bts,sl->btl', active, ct['matrix'])
        aggregate = lambda values: torch.einsum('btsc,sl->btlc', values * active[..., None], ct['matrix']) / counts.clamp_min(1)[..., None]
        links, true_links = aggregate(predicted), aggregate(reference)
        affected = torch.einsum('bts,sl->btl', target.float(), ct['matrix']) > 0
        good_n = (counts > 0) & affected
        n = links[..., 1] / links[..., 0].clamp_min(1) * ct['lengths']
        true_n = true_links[..., 1] / true_links[..., 0].clamp_min(1) * ct['lengths']
        losses['N'] = (n - true_n)[good_n].abs().sum() / (true_n[good_n].abs().sum() + 1.)
        transition = training_transitions(d, context)
        good_dn = torch.as_tensor(np.stack([transition[s:s+length-1] for s in starts]), device=device)
        dn, true_dn = torch.diff(n, dim=1), torch.diff(true_n, dim=1)
        losses['dN'] = (dn - true_dn)[good_dn].abs().sum() / (true_dn[good_dn].abs().sum() + 1.)
        if LOSS_WEIGHTS['FD'] > 0:
            density = links[..., 1] / links[..., 0].clamp_min(1)
            critical = ct['capacity'] / ct['free']
            wave = ct['capacity'] / (ct['jam'] - critical).clamp_min(1e-6)
            qfd = torch.where(density <= critical, ct['free'] * density, wave * (ct['jam'] - density)).clamp_min(0)
            error = ((links[..., 1] - qfd) / ct['link_lanes'])[good_n]
            losses['FD'] = error.abs().sum() / ((links[..., 1] / ct['link_lanes'])[good_n].detach().abs().sum() + 1.)
    total = state + physics_factor * sum(LOSS_WEIGHTS[k] * losses[k] for k in LOSS_WEIGHTS)
    if not torch.isfinite(total):
        raise FloatingPointError('학습 손실에 NaN/Inf가 있습니다.')
    return total, {k: float(v.detach()) for k, v in losses.items()}


def evaluate(entries, contexts, tensors, model, period, alphas, folder):
    started, records = time.perf_counter(), []
    selected = [e for e in entries if e['period'] == period]
    if not selected:
        raise ValueError(f'평가 날짜가 없습니다: {period}')
    for number, entry in enumerate(selected, 1):
        d, context = load_day(entry['path']), contexts[entry['panel']]
        output = maginet_predict(model, d, tensors[entry['panel']])
        ti, si = d['ti'], d['si']
        truth, base, prediction = d['truth'][ti, si] * SCALE, d['base'][ti, si] * SCALE, output[ti, si] * SCALE
        for alpha in alphas:
            mixed = (1 - alpha) * base + alpha * prediction
            records.append(dict(candidate=f'alpha_{float(alpha)!r}', alpha=float(alpha),
                                panel=entry['panel'], date=entry['date'],
                                regime=entry['regime'], **state_totals(truth, mixed), **physics_totals(mixed, d, context)))
        if number == 1 or number % 20 == 0 or number == len(selected):
            print(f'평가 {period}: {number}/{len(selected)}일', flush=True)
    macro, grouped, daily = summarize(records)
    folder.mkdir(parents=True, exist_ok=True)
    for name, frame in (('macro', macro), ('panel_regime', grouped), ('daily', daily)):
        frame.to_csv(folder / f'{period}_{name}.csv', index=False)
    if (grouped.lwr_count == 0).any():
        print('주의: LWR 대리 평가 대상이 없는 그룹이 있습니다. panel_regime의 lwr_count를 확인해 주세요.')
    return macro, grouped, daily, time.perf_counter() - started


def run_signature(contexts, entries, normalization):
    settings = dict(architecture='maginet-task13-residual-v2',
                    input_normalization={k: normalization[k] for k in ('mode', 'mean', 'std')},
                    output_scale=SCALE.tolist(), aux_names=list(AUX_NAMES), seed=SEED, window=WINDOW, stride=STRIDE,
                    width=WIDTH, blocks=BLOCKS, heads=HEADS, cheb_k=CHEB_K, graph_dim=GRAPH_DIM,
                    dropout=DROPOUT, mgl=MGL_THRESHOLD, batch=BATCH_SIZE, updates=UPDATES_PER_GROUP,
                    lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                    warmup=WARMUP_EPOCHS, ramp=PHYSICS_RAMP_EPOCHS, loss_weights=LOSS_WEIGHTS,
                    select_weights=SELECT_WEIGHTS, periods=PERIODS, release=str(RELEASE_ROOT.resolve()),
                    training_dates={n: sorted(e['date'] for e in entries if e['panel'] == n and e['period'] == 'fit')
                                    for n in contexts},
                    panels={name: dict(stations=c['targets'].stations, links=c['targets'].links,
                                      station_codes=c['station_codes'].tolist(), panel_code=c['panel_code'],
                                      family=c['family_code'], direction=c['direction_code'], mode=c['mode'],
                                      dates=c['dates'], profile_hash=hashlib.sha256(c['profile'].tobytes()).hexdigest())
                            for name, c in contexts.items()})
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(), settings


def save_checkpoint(payload, path):
    temporary = path.with_suffix('.pt.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def optimizer_step(model, optimizer, scaler):
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=not scaler.is_enabled())
    scaler.step(optimizer)
    scaler.update()


def train_maginet(entries, contexts, tensors, folder, epochs=EPOCHS, resume=RESUME):
    reset_seed()
    folder.mkdir(parents=True, exist_ok=True)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    if not resume and (best_path.exists() or last_path.exists()):
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    normalization = fit_input_normalization(entries, folder)
    signature, settings = run_signature(contexts, entries, normalization)
    station_count = max(int(c['station_codes'].max()) for c in contexts.values()) + 1
    panel_count = max(c['panel_code'] for c in contexts.values()) + 1
    family_count = max(c['family_code'] for c in contexts.values()) + 1
    model = MagiNetTask13(station_count, panel_count, family_count, normalization).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    start_epoch, best, bad_epochs, best_epoch, history = 1, -float('inf'), 0, 0, []
    if resume:
        if not last_path.exists() or not best_path.exists():
            raise FileNotFoundError('재개하려면 last_maginet.pt와 best_maginet.pt가 모두 필요합니다.')
        state = torch.load(last_path, map_location=DEVICE, weights_only=True)
        if state['signature'] != signature:
            raise ValueError('설정·패널·학습 날짜가 달라 재개할 수 없습니다. REPORT_DIR를 바꾸어 새로 학습해 주세요.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best, bad_epochs, best_epoch, history = state['epoch'] + 1, state['best'], state['bad_epochs'], state['best_epoch'], state['history']
        torch.set_rng_state(state['rng_cpu'].cpu())
        if USE_AMP and state['rng_cuda']:
            torch.cuda.set_rng_state_all([v.cpu() for v in state['rng_cuda']])
        if epochs < start_epoch:
            raise ValueError(f'EPOCHS를 마지막 완료 epoch {start_epoch-1}보다 크게 지정해 주세요.')
    elif best_path.exists() or last_path.exists():
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    fit_entries = [e for e in entries if e['period'] == 'fit']
    pools = {(name, r): [e for e in fit_entries if e['panel'] == name and e['regime'] == r]
             for name in contexts for r in range(3)}
    if any(not p for p in pools.values()):
        raise ValueError('공동 학습에 panel×regime별 날짜가 모두 필요합니다.')
    # 재개 설정과 학습 그룹을 확인한 뒤 해당 실행 폴더에만 저장합니다.
    (folder / 'config.json').write_text(json.dumps(settings, indent=2, ensure_ascii=False))
    (folder / 'input_normalization.json').write_text(json.dumps(normalization, indent=2, ensure_ascii=False))
    for name, context in contexts.items():
        np.save(folder / f'fit_profile-{name}.npy', context['profile'])
    pd.DataFrame(entries).to_csv(folder / 'cache_manifest.csv', index=False)
    direction_count = {n: sum(p.rsplit('_', 1)[0] == n.rsplit('_', 1)[0] for p in contexts) for n in contexts}
    began = time.perf_counter()
    prior_elapsed = history[-1]['elapsed_seconds'] if history else 0.
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        groups = list(pools)
        rng.shuffle(groups)
        totals = dict(State=0., N=0., dN=0., FD=0.)
        updates, train_started = 0, time.perf_counter()
        physics_factor = min(1., max(0., (epoch - WARMUP_EPOCHS) / max(PHYSICS_RAMP_EPOCHS, 1)))
        for name, regime in groups:
            pool = pools[(name, regime)]
            probabilities = np.array([e['targets'] for e in pool], np.float64)
            probabilities /= probabilities.sum()
            for step in range(UPDATES_PER_GROUP):
                if step % 4 == 0:
                    entry = pool[int(rng.choice(len(pool), p=probabilities))]
                    d = load_day(entry['path'])
                    starts = [s for s in window_starts(len(d['values'])) if d['target'][s:s+WINDOW].any()]
                    if not starts:
                        raise ValueError('대상 셀이 있는 학습 창이 없습니다.')
                chunk = rng.choice(starts, size=BATCH_SIZE, replace=True).tolist()
                batch = window_batch(d, chunk)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    output = model(**batch, ct=tensors[name])
                loss, terms = joint_loss(output.float(), d, chunk, contexts[name], tensors[name], physics_factor)
                # 도로군마다 방향 수가 달라도 누적 가중치가 같도록 맞춥니다.
                scaler.scale(loss / direction_count[name]).backward()
                optimizer_step(model, optimizer, scaler)
                updates += 1
                for k in totals:
                    totals[k] += terms[k]
            print(f'epoch {epoch}/{epochs}: {name}/{REGIMES[regime]} 학습 완료', flush=True)
        train_seconds = time.perf_counter() - train_started
        macro, _, _, eval_seconds = evaluate(entries, contexts, tensors, model, 'early', (1.,), folder / 'early')
        score = float(macro.iloc[0].selection_proxy)
        if not np.isfinite(score):
            raise FloatingPointError('체크포인트 선택 점수가 유한하지 않습니다.')
        scheduler.step(score)
        if score > best + 1e-7:
            best, bad_epochs, best_epoch = score, 0, epoch
            save_checkpoint(dict(model=model.state_dict(), signature=signature, epoch=epoch, score=score), best_path)
        else:
            bad_epochs += 1
        row = dict(epoch=epoch, **{f'loss_{k}': v / updates for k, v in totals.items()},
                   early_State=float(macro.iloc[0].State), early_selection_proxy=score,
                   physics_factor=physics_factor, lr=optimizer.param_groups[0]['lr'],
                   train_seconds=train_seconds, eval_seconds=eval_seconds,
                   elapsed_seconds=prior_elapsed + time.perf_counter() - began)
        history.append(row)
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        save_checkpoint(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                        scaler=scaler.state_dict(), signature=signature, epoch=epoch, best=best,
                        bad_epochs=bad_epochs, best_epoch=best_epoch, history=history,
                        rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all() if USE_AMP else []), last_path)
        print({'epoch': epoch, 'State': row['early_State'], 'selection_proxy': score,
               'best_epoch': best_epoch, 'physics_factor': physics_factor}, flush=True)
        if bad_epochs >= PATIENCE and epoch >= WARMUP_EPOCHS + PHYSICS_RAMP_EPOCHS + 2:
            break
    best_state = torch.load(best_path, map_location=DEVICE, weights_only=True)
    if best_state['signature'] != signature:
        raise ValueError('저장된 best 체크포인트의 설정이 다릅니다.')
    model.load_state_dict(best_state['model'])
    model.eval()
    print({'best_epoch': best_state['epoch'], 'best_early_selection_proxy': best_state['score']})
    return model, pd.DataFrame(history)


## 5. validation/private CSV 생성

1월 후반에서 선택한 **하나의 공통 α**를 모든 패널·regime·두 출력에 적용합니다. `α=0`은 보간, `α=1`은 MagiNet 단독입니다. 대상 외의 공개 관측은 물리 진단에 그대로 사용합니다. CSV는 공식 템플릿의 행 순서와 키를 유지하고 차로당 예측을 관측소 총 교통량으로 바꿉니다. 패널별 Task 1 CSV가 Task 3의 입력도 됩니다.

In [ ]:
def predict_split(contexts, model, tensors, alpha, split, folder):
    if split not in ('validation', 'private') or not 0 <= alpha <= 1:
        raise ValueError('추론 split 또는 alpha가 잘못되었습니다.')
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, c in contexts.items():
        template_path = c['release'] / 'task1' / name / split / 'sample_submission_state.csv'
        template = pd.read_csv(template_path, dtype={k: str for k in KEYS})
        if template.empty or template.duplicated(KEYS).any() or set(template.panel) != {name}:
            raise ValueError(f'잘못된 제출 템플릿: {template_path}')
        files = dated_files(c['release'] / 'corridors' / name / split / 'mainline_states_masked')
        if not files:
            raise FileNotFoundError(f'공개 masked 입력이 없습니다: {name}/{split}')
        first = StationDay.load(next(iter(files.values())), c['panel'])
        target_index = TargetIndex(c['release'], name, first, split)
        if (first.station_ids, first.station_links) != (c['targets'].stations, c['targets'].links):
            raise ValueError(f'추론 관측소 구성이 다릅니다: {name}')
        out = template.copy()
        out[['speed_kmh', 'flow_vph']] = np.nan
        for number, (date, group) in enumerate(template.groupby(template.timestamp.str[:10], sort=False), 1):
            if date not in files:
                raise ValueError(f'추론 날짜가 누락되었습니다: {name}/{date}')
            d = make_day(c, date, split=split, target_index=target_index, input_files=files)
            ti, si = d['ti'], d['si']
            base = d['base'][ti, si]
            neural = maginet_predict(model, d, tensors[name])[ti, si] if alpha > 0 else base
            values = ((1 - alpha) * base + alpha * neural) * SCALE
            local = pd.DataFrame(dict(timestamp=d['timestamps'][ti], station_id=np.asarray(c['targets'].stations)[si],
                                      speed_kmh=values[:, 0], flow_vph=values[:, 1] * c['lanes'][si]))
            aligned = group[['timestamp', 'station_id']].merge(local, on=['timestamp', 'station_id'],
                                                               how='left', validate='one_to_one', sort=False)
            if aligned[['speed_kmh', 'flow_vph']].isna().any().any():
                raise ValueError(f'추론 대상 키가 일치하지 않습니다: {name}/{date}')
            out.loc[group.index, ['speed_kmh', 'flow_vph']] = aligned[['speed_kmh', 'flow_vph']].to_numpy()
            if number == 1 or number % 10 == 0:
                print(f'CSV 예측 {name}/{split}: {number}일', flush=True)
        values = out[['speed_kmh', 'flow_vph']].to_numpy()
        if not out[KEYS].equals(template[KEYS]) or not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f'출력 값 또는 템플릿 키가 잘못되었습니다: {name}')
        path = folder / f'task1-{name}-{split}.csv'
        temporary = path.with_suffix('.csv.tmp')
        out.to_csv(temporary, index=False)
        temporary.replace(path)
        paths.append(str(path))
    return paths


## 6. 작은 합성 입력 자체 검사

아래 셀은 데이터 파일 없이 결측 토큰, 전부 결측인 Attention, 가변 노드·시간 길이, batch 간 독립성, Chebyshev 재귀, 겹치는 창 예측, 물리 손실, ns/us 시간 간격, private 분할 허용, 소수 α 보존, 기존 파일 보호, 초기 보간 출력, 양·음 보정, fit 관측만의 평균·표준편차와 실제 표준화 입력, 경로·mtime 변경 후 저장 통계 재사용과 입력 변경 시 재계산, 표준화 checkpoint, NPZ·ZIP 복원과 잘못된 캐시/ZIP 거부를 확인합니다. CUDA 환경에서는 수치 넘침 시 업데이트를 건너뛰고 scale을 낮추는지도 확인합니다. **작성 환경에서는 이 실행 검사를 수행하지 않았습니다.** Colab에서 이 셀이 통과한 뒤 다음 학습 셀로 진행해 주세요. 통과해도 실제 데이터·GPU 검증을 대신하지 않습니다.

In [ ]:
def run_self_checks():
    reset_seed()
    device = torch.device('cpu')
    identity = dict(mode='fixed', mean=[0., 0., 0.], std=[1., 1., 1.])
    model = MagiNetTask13(4, 1, 1, identity).to(device).eval()

    def fixture(stations, length):
        graph = np.eye(stations, dtype=np.float32)
        for i in range(stations - 1):
            graph[i, i+1] = graph[i+1, i] = 1
        names = [f'l{i}' for i in range(stations)]
        panel = PanelInfo('demo_N', names, {n: i for i, n in enumerate(names)},
                          np.full(stations, 100., np.float32), np.full(stations, 2., np.float32),
                          np.ones(stations, np.float32), {},
                          np.full(stations, 3600., np.float32), np.full(stations, 200., np.float32))
        context = dict(panel=panel, mapping=np.arange(stations), station_codes=np.arange(stations),
                       panel_code=0, family_code=0, direction_code=0, lanes=np.full(stations, 2., np.float32),
                       station_graph=graph, mode='A', topology_nodes=set(range(stations)))
        truth = np.broadcast_to(np.array([3., 1.], np.float32), (length, stations, 2)).copy()
        target = np.zeros((length, stations), bool)
        target[2:6, 0] = True
        values = np.concatenate((truth, np.full((length, stations, 1), .1, np.float32)), -1)
        visible = np.ones_like(values, bool)
        visible[target] = False
        values[~visible] = 0
        physical = truth * np.array([25., 1200.], np.float32)
        physical[target] = 0
        ti, si = np.nonzero(target)
        timestamps = pd.date_range('2031-01-01', periods=length, freq='5min', tz='UTC').strftime('%Y-%m-%dT%H:%M:%SZ').tolist()
        d = dict(values=values, visible=visible, aux=np.zeros((length, stations, len(AUX_NAMES)), np.float32),
                 slot=np.arange(length, dtype=np.int64), weekday=np.zeros(length, np.int64),
                 regime=np.array(0, np.int32), base=truth.copy(), truth=truth, target=target,
                 active=np.ones_like(target), observed=physical, ti=ti, si=si,
                 timestamps=np.asarray(timestamps), ramp_valid=np.ones((length, stations), bool),
                 consecutive=consecutive_five_minutes(timestamps))
        return d, context, tensor_context(context, device)

    times = pd.to_datetime(['2031-01-01T00:00:00Z', '2031-01-01T00:05:00Z', '2031-01-01T00:15:00Z'], utc=True)
    for unit in ('ns', 'us'):
        np.testing.assert_array_equal(consecutive_five_minutes(times.as_unit(unit)), [True, False])
    scores = torch.randn(2, 3, 4)
    assert torch.equal(masked_softmax(scores, torch.zeros_like(scores, dtype=torch.bool)), torch.zeros_like(scores))
    d, c, ct = fixture(3, 12)
    batch = window_batch(d, [0], device)
    with torch.no_grad():
        assert torch.equal(model(**batch, ct=ct), batch['base'])
        model.speed_head[-1].bias.fill_(-.25)
        model.flow_head[-1].bias.fill_(.25)
        shifted = model(**batch, ct=ct)
        assert torch.allclose(shifted[..., 0], batch['base'][..., 0] - .25)
        assert torch.allclose(shifted[..., 1], batch['base'][..., 1] + .25)
        model.speed_head[-1].bias.fill_(-100.)
        assert (model(**batch, ct=ct)[..., 0] == 0).all()
        for head in (model.speed_head, model.flow_head):
            nn.init.normal_(head[-1].weight, std=.01)
            nn.init.zeros_(head[-1].bias)
    for stations, length in ((3, 12), (4, 17)):
        d, c, ct = fixture(stations, length)
        batch = window_batch(d, [0, 0], device)
        with torch.no_grad():
            prediction = model(**batch, ct=ct)
            altered = dict(batch)
            altered['values'] = torch.where(batch['visible'], batch['values'], torch.full_like(batch['values'], 999.))
            assert torch.allclose(prediction, model(**altered, ct=ct), atol=1e-6, rtol=1e-6)
            separate = torch.cat([model(**{k: v[i:i+1] for k, v in batch.items()}, ct=ct) for i in range(2)])
            assert torch.allclose(prediction, separate, atol=1e-5, rtol=1e-5)
            invisible = dict(batch)
            invisible['visible'] = torch.zeros_like(batch['visible'])
            assert torch.isfinite(model(**invisible, ct=ct)).all()
        assert prediction.shape == (2, length, stations, 2) and torch.isfinite(prediction).all() and (prediction >= 0).all()
        supports = chebyshev_supports(c['station_graph'], order=3)
        assert np.allclose(supports[2], 2 * supports[1] @ supports[1] - supports[0], atol=1e-5)
        completed, counts, affected = completed_link_fields(d['truth'][d['ti'], d['si']] * SCALE, d, c)
        assert np.allclose(completed[..., 0], 75.) and np.allclose(completed[..., 1], 1200.)
        assert np.all(counts == 1) and affected.sum() == d['target'].sum()
        model.zero_grad(set_to_none=True)
        output = model(**batch, ct=ct)
        loss, terms = joint_loss(output, d, [0, 0], c, ct, physics_factor=1.)
        loss.backward()
        assert np.isfinite(list(terms.values())).all()
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    d, c, ct = fixture(3, min(WINDOW + 5, 288))
    coverage = np.zeros(len(d['values']), int)
    for s in window_starts(len(coverage)):
        coverage[s:s+WINDOW] += 1
    assert (coverage > 0).all()
    assert np.allclose(maginet_predict(model, d, ct, 1), maginet_predict(model, d, ct, 2), atol=1e-5, rtol=1e-5)
    with tempfile.TemporaryDirectory(prefix='maginet-check-') as directory:
        folder = Path(directory)
        for split in ('validation', 'private'):
            assert predict_split({}, model, {}, .125, split, folder / split) == []
        entries = []
        for regime in range(3):
            sample = dict(d, regime=np.array(regime, np.int32))
            sample_path = folder / f'r{regime}.npz'
            np.savez(sample_path, **sample)
            entries.append(dict(panel='demo_N', date=f'2031-01-0{regime+1}', period='check',
                                regime=regime, targets=int(sample['target'].sum()), path=str(sample_path)))
        macro, grouped, daily, _ = evaluate(entries, {'demo_N': c}, {'demo_N': ct}, model,
                                            'check', (.121, .124, .125), folder / 'evaluation')
        assert len(macro) == 3 and set(macro.alpha) == {.121, .124, .125}
        assert grouped.candidate.nunique() == 3 and daily.candidate.nunique() == 3

        norm_dir = folder / 'normalization'
        first_path = folder / 'fit.npz'
        normalization_day = dict(d, values=d['values'].copy(), aux=d['aux'].copy())
        ramp = np.linspace(0., 1., len(d['values']), dtype=np.float32)[:, None, None]
        varied = d['values'] + ramp * np.array([2., .8, .5], np.float32)
        normalization_day['values'] = np.where(d['visible'], varied, 9999.)
        normalization_day['aux'][..., 8:11] = [3., 1., .1]
        np.savez_compressed(first_path, values=normalization_day['values'], visible=d['visible'])
        norm_entries = [dict(panel='demo_N', date='2031-01-01', period='fit', path=str(first_path))]
        # 결측 placeholder와 평가 기간의 극단값은 fit 통계에 포함하지 않습니다.
        validation_path = folder / 'early.npz'
        np.savez_compressed(validation_path, values=np.full_like(d['values'], 9999.), visible=d['visible'])
        norm_entries.append(dict(panel='demo_N', date='2031-01-02', period='early', path=str(validation_path)))
        normalizer = fit_input_normalization(norm_entries, norm_dir, mode='standard')
        for channel in range(3):
            observed = normalization_day['values'][..., channel][d['visible'][..., channel]].astype(np.float64)
            assert np.isclose(normalizer['mean'][channel], observed.mean())
            assert np.isclose(normalizer['std'][channel], observed.std(ddof=0))
            assert not np.isclose(normalizer['std'][channel], 1.)
            assert normalizer['count'][channel] == len(observed)
        assert fit_input_normalization(norm_entries, norm_dir, mode='standard') == normalizer

        # 런타임 재시작처럼 경로·mtime이 달라도 통계 재계산 없이 재사용합니다.
        from os import utime
        from unittest import mock
        relocated = folder / 'new-runtime' / 'fit.npz'
        relocated.parent.mkdir()
        shutil.copyfile(first_path, relocated)
        stamp = first_path.stat().st_mtime_ns + 10_000_000_000
        utime(relocated, ns=(stamp, stamp))
        moved_entries = [dict(e, path=str(relocated)) if e['period'] == 'fit' else e for e in norm_entries]
        with mock.patch.object(np, 'load', side_effect=AssertionError('저장 통계를 다시 계산했습니다.')):
            assert fit_input_normalization(moved_entries, norm_dir, mode='standard') == normalizer
        changed = normalization_day['values'].copy()
        changed[..., 0] += np.where(d['visible'][..., 0], .5, 0.)
        np.savez_compressed(relocated, values=changed, visible=d['visible'])
        updated = fit_input_normalization(moved_entries, norm_dir, mode='standard')
        assert updated['source_signature'] != normalizer['source_signature']
        assert np.isclose(updated['mean'][0], normalizer['mean'][0] + .5)
        fixed = fit_input_normalization(norm_entries, norm_dir, mode='fixed')
        assert fixed['mean'] == [0., 0., 0.] and fixed['std'] == [1., 1., 1.]
        normalized_model = MagiNetTask13(4, 1, 1, normalizer).eval()
        captured = {}
        def capture_encoder_input(module, args):
            captured.update(values=args[0].detach().clone(), aux=args[2].detach().clone(),
                            base=args[3].detach().clone())
        hook = normalized_model.AMSTenc.register_forward_pre_hook(capture_encoder_input)
        normalization_batch = window_batch(normalization_day, [0], device)
        try:
            with torch.no_grad():
                initial = normalized_model(**normalization_batch, ct=ct)
        finally:
            hook.remove()
        mean = torch.as_tensor(normalizer['mean'], dtype=torch.float32)
        std = torch.as_tensor(normalizer['std'], dtype=torch.float32)
        expected_values = torch.where(normalization_batch['visible'],
                                      (normalization_batch['values'] - mean) / std, 0.)
        assert torch.allclose(captured['values'], expected_values, atol=1e-6, rtol=1e-6)
        assert torch.allclose(captured['base'], (normalization_batch['base'] - mean[:2]) / std[:2])
        assert torch.allclose(captured['aux'][..., 8:11], (normalization_batch['aux'][..., 8:11] - mean) / std)
        assert torch.equal(initial, normalization_batch['base'])
        normalization_checkpoint = folder / 'normalization.pt'
        save_checkpoint({'model': normalized_model.state_dict()}, normalization_checkpoint)
        restored = MagiNetTask13(4, 1, 1, identity).eval()
        restored.load_state_dict(torch.load(normalization_checkpoint, weights_only=True)['model'])
        assert torch.equal(restored.input_mean, normalized_model.input_mean)
        assert torch.equal(restored.input_std, normalized_model.input_std)

        from types import SimpleNamespace
        date = str(d['timestamps'][0])[:10]
        ti, si = d['ti'], d['si']
        cells = d['slot'][ti] * 3 + si
        target_index = SimpleNamespace(stations=['s0', 's1', 's2'], links=c['panel'].links,
                                       archive={date: cells}, regimes={date: 'R1'})
        c['targets'] = target_index
        validate_cache_targets(d, c, date)
        bad = dict(d, regime=np.array(1, np.int32))
        try:
            validate_cache_targets(bad, c, date)
        except ValueError:
            pass
        else:
            raise AssertionError('다른 regime의 캐시를 허용했습니다.')
        existing = folder / 'cache' / 'demo_N' / 'fit' / f'{date}.npz'
        existing.parent.mkdir(parents=True)
        np.savez_compressed(existing, **d)
        c['dates'] = {p: ([date] if p == 'fit' else []) for p in PERIODS}
        before = existing.stat().st_mtime_ns
        reused = cache_context(c, folder / 'cache', periods=('fit',))
        assert len(reused) == 1 and reused[0]['targets'] == int(d['target'].sum())
        assert existing.stat().st_mtime_ns == before
        if CACHE_MODE == 'reuse':
            try:
                cache_context(c, folder / 'missing', periods=('fit',))
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('reuse 모드에서 누락 캐시를 다시 생성했습니다.')
        loaded = load_day(str(existing))
        assert np.array_equal(loaded['base'], d['base'])
        archive_path = folder / 'saved.zip'
        with zipfile.ZipFile(archive_path, 'w') as archive:
            archive.write(existing, f'preprocessed/demo_N/fit/{date}.npz')
        if shutil.disk_usage(folder).free >= existing.stat().st_size + 2 * 2**30:
            imported, _ = restore_input_cache(['demo_N'], archive_path, folder / 'restored')
            assert np.array_equal(load_day(str(imported / 'demo_N' / 'fit' / f'{date}.npz'))['base'], d['base'])
            assert restore_input_cache(['demo_N'], archive_path, folder / 'restored')[0] == imported
        hostile = folder / 'hostile.zip'
        with zipfile.ZipFile(hostile, 'w') as archive:
            archive.writestr('../outside.npz', b'bad')
        try:
            restore_input_cache(['demo_N'], hostile, folder / 'blocked')
        except ValueError:
            pass
        else:
            raise AssertionError('ZIP 경로 이탈을 허용했습니다.')
        c.update(targets=target_index,
                 profile=np.ones((1, 3, 3), np.float32), dates={p: [] for p in PERIODS})
        run_folder = folder / 'protected_run'
        run_folder.mkdir()
        saved = [run_folder / 'config.json', run_folder / 'fit_profile-demo_N.npy',
                 run_folder / 'cache_manifest.csv', run_folder / 'best_maginet.pt']
        for p in saved:
            p.write_bytes(b'previous experiment')
        save_checkpoint({'signature': 'different settings'}, run_folder / 'last_maginet.pt')
        for resume, expected_error in ((True, ValueError), (False, FileExistsError)):
            try:
                train_maginet([dict(e, period='fit') for e in entries], {'demo_N': c}, {'demo_N': ct}, run_folder, epochs=1, resume=resume)
            except expected_error:
                pass
            else:
                raise AssertionError('기존 실행 파일 보호 검사가 실패했습니다.')
            assert all(p.read_bytes() == b'previous experiment' for p in saved)

    if torch.cuda.is_available():
        probe = nn.Linear(1, 1, bias=False, device='cuda')
        optimizer = torch.optim.SGD(probe.parameters(), lr=.1)
        scaler = torch.amp.GradScaler('cuda')
        scaler.scale(probe(torch.ones(1, 1, device='cuda')).sum()).backward()
        probe.weight.grad.fill_(float('inf'))
        previous_weight, previous_scale = probe.weight.detach().clone(), scaler.get_scale()
        optimizer_step(probe, optimizer, scaler)
        assert torch.equal(probe.weight, previous_weight) and scaler.get_scale() < previous_scale
    else:
        print('FP16 수치 넘침 복구 검사는 CUDA 환경에서 실행됩니다.')
    load_day.cache_clear()
    reset_seed()
    print('자체 검사 통과: 잔차 ± / 표준화·checkpoint / NPZ·ZIP / mask / 형태 / Chebyshev / 물리 손실 / 창 결합 / 시간 단위 / private / alpha / 파일 보호')

run_self_checks()


## 7. 첫 패널에서 먼저 점수 확인

처음 선택한 패널 하나, regime별 최대 4개 학습일, 2 epoch로 데이터·MagiNet·평가 흐름을 먼저 확인합니다. 과거 평균은 설정된 전체 fit 구간에서 계산합니다. 이 단계는 State 준비 학습이며 전체 패널 점수가 아닙니다. 첫 점수까지의 시간에는 해당 패널의 전처리·캐시·학습·평가가 포함됩니다.

저장 ZIP을 런타임 로컬 디스크에 복원하거나 지정된 NPZ 폴더를 읽으며 진행률을 출력합니다. 기본 reuse 모드에서는 입력 전처리와 fit 과거 평균 생성을 생략합니다. 첫 단계의 캐시는 전체 학습에도 재사용하고 모델은 전체 학습 시작 시 새로 만듭니다. `RESUME=True`이면 첫 점검 학습을 생략합니다. 노트북 전체 재실행 시 기존 체크포인트를 덮어쓰지 않으므로 `RESUME` 또는 새 `REPORT_DIR`를 지정해 주세요.

In [ ]:
RUN_STARTED = time.perf_counter()
NAMES = available_panels(RELEASE_ROOT)
CACHE_ROOT, CACHE_CONTAINER = restore_input_cache(NAMES, SAVED_CACHE_ZIP, CACHE_DIR)
CONTEXTS = {}
FIRST_SCORE_SECONDS = None
print({'panels': NAMES, 'cache': str(CACHE_ROOT)})

if RUN_FIRST_PANEL and not RESUME:
    name = NAMES[0]
    c = prepare_panel(RELEASE_ROOT, name, 0, 0)
    CONTEXTS[name] = c
    attach_vocabulary(CONTEXTS)
    first_dates = dict(c['dates'])
    chosen = []
    for regime in REGIMES:
        pool = [d for d in c['dates']['fit'] if c['targets'].regimes[d] == regime]
        indices = np.linspace(0, len(pool)-1, min(FIRST_DAYS_PER_REGIME, len(pool)), dtype=int)
        chosen += [pool[i] for i in indices]
    first_dates['fit'] = sorted(chosen)
    first_entries = cache_context(c, CACHE_ROOT, first_dates)
    first_tensors = {name: tensor_context(c)}
    first_model, first_history = train_maginet(first_entries, CONTEXTS, first_tensors,
                                              REPORT_DIR / 'first_panel', epochs=FIRST_EPOCHS, resume=False)
    first_macro, first_groups, first_daily, _ = evaluate(first_entries, CONTEXTS, first_tensors, first_model,
                                                        'blend', (0., 1.), REPORT_DIR / 'first_panel')
    FIRST_SCORE_SECONDS = time.perf_counter() - RUN_STARTED
    display(first_macro)
    print({'scope': f'첫 패널 {name} 진단', 'time_to_first_score_seconds': FIRST_SCORE_SECONDS})
    del first_model, first_tensors
    if USE_AMP:
        torch.cuda.empty_cache()
else:
    print('첫 패널 점검 생략: 전체 학습 단계로 진행합니다.')


## 8. 전체 패널 공동 학습

한 batch는 같은 패널·날짜의 시간 창으로 구성하므로 노드 수가 다른 패널도 처리합니다. 모든 패널이 하나의 AMSTenc·MASTdec를 공유합니다. 패널×regime별로 같은 업데이트 횟수를 배정하고 같은 도로군의 방향 수로 손실을 나눕니다. R1·R2·R3는 입력 regime 임베딩으로 구분합니다.

1월 전반의 대리 선택 점수로 체크포인트를 고릅니다. 이후 1월 후반으로 α를 한 번 선택합니다. `best_maginet.pt`, 재개용 `last_maginet.pt`, 설정·학습 기록·패널별 fit 평균·캐시 목록이 Drive의 `full/` 실행 폴더에 저장됩니다. 첫 점검 자료는 `first_panel/`에 저장됩니다. 입력 표준화 통계도 `input_normalization.json`과 체크포인트에 저장됩니다. 재개 시 같은 데이터와 설정이 필요하며 `EPOCHS`를 마지막 완료 epoch보다 크게 지정해 주세요.

In [ ]:
ENTRIES = []
offset = 0
for panel_code, name in enumerate(NAMES):
    if name not in CONTEXTS:
        CONTEXTS[name] = prepare_panel(RELEASE_ROOT, name, panel_code, offset)
    c = CONTEXTS[name]
    c['panel_code'] = panel_code
    c['station_codes'] = np.arange(offset, offset + len(c['targets'].stations))
    offset += len(c['targets'].stations)
    ENTRIES.extend(cache_context(c, CACHE_ROOT))
attach_vocabulary(CONTEXTS)
TENSORS = {name: tensor_context(c) for name, c in CONTEXTS.items()}
MODEL, HISTORY = train_maginet(ENTRIES, CONTEXTS, TENSORS, REPORT_DIR / 'full')
display(HISTORY.tail())

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(HISTORY.epoch, HISTORY.early_State, label='early State')
ax.plot(HISTORY.epoch, HISTORY.early_selection_proxy, label='early selection proxy')
ax.set(xlabel='epoch', ylabel='score', ylim=(0, 1))
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / 'training_curve.png', dpi=140)
plt.show()


## 9. 보간 대비 α 선택과 결과 저장

State만 좋아지고 N·ΔN이 악화되는지 `blend_panel_regime.csv`와 `blend_daily.csv`에서도 확인해 주세요. 표의 방향 평균·도로군 평균·regime 평균은 공식 집계 구조를 따릅니다. 두 출력에 같은 α를 써서 최종 혼합 예측으로 물리 진단을 다시 계산합니다. α는 반올림하지 않고 숫자 열로 보존합니다. 대리 선택식이 같으면 더 작은 α를 고릅니다. 보간이 이기면 α=0을 유지합니다.

In [ ]:
MACRO, PANEL_REGIME, DAILY, SELECTION_SECONDS = evaluate(ENTRIES, CONTEXTS, TENSORS, MODEL,
                                                        'blend', ALPHAS, REPORT_DIR)
MACRO = MACRO.sort_values(['selection_proxy', 'alpha'], ascending=[False, True]).reset_index(drop=True)
SELECTED_ALPHA = float(MACRO.iloc[0].alpha)
MACRO.to_csv(REPORT_DIR / 'blend_macro.csv', index=False)
selection = dict(alpha=SELECTED_ALPHA, candidate=MACRO.iloc[0].candidate,
                 selection_proxy=float(MACRO.iloc[0].selection_proxy),
                 criterion=SELECT_WEIGHTS, selected_on='2031-01-16~31',
                 checkpoint=str(REPORT_DIR / 'full' / 'best_maginet.pt'))
(REPORT_DIR / 'selected_alpha.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False))
display(MACRO)
display(PANEL_REGIME[PANEL_REGIME.candidate == selection['candidate']][
    ['panel', 'regime', 'State', 'N_MAE_proxy', 'DN_MAE_proxy', 'FD_proxy', 'LWR_proxy', 'lwr_count']])
print({'selected_alpha': SELECTED_ALPHA, 'total_seconds_through_selection': time.perf_counter() - RUN_STARTED,
       'first_panel_time_to_score_seconds': FIRST_SCORE_SECONDS,
       'preparation_seconds': sum(c['preparation_seconds'] for c in CONTEXTS.values()),
       'cache_seconds': sum(c['cache_seconds'] for c in CONTEXTS.values())})


## 10. 선택값 고정 후 2월 추가 진단 (선택)

`RUN_FEBRUARY=True`일 때만 실행합니다. 모델·α·입력 표준화는 다시 선택하지 않습니다. 저장된 2월 NPZ도 재사용하며 reuse 모드에서 없으면 중단합니다. 앞선 실험에서 이미 사용된 2월은 추가 진단으로 표시하며 새로운 독립 holdout이라고 부르지 않습니다.

In [ ]:
if RUN_FEBRUARY:
    february_entries = []
    for name, c in CONTEXTS.items():
        dates = c['dates']['february']
        if not dates or {c['targets'].regimes.get(d) for d in dates} != set(REGIMES):
            raise ValueError(f'2월 R1·R2·R3 날짜가 부족합니다: {name}')
        february_entries.extend(cache_context(c, CACHE_ROOT, periods=('february',)))
    FEB_MACRO, FEB_GROUPS, FEB_DAILY, _ = evaluate(february_entries, CONTEXTS, TENSORS, MODEL, 'february',
                                                 tuple(sorted({0., SELECTED_ALPHA})), REPORT_DIR)
    display(FEB_MACRO)
else:
    print('2월 추가 진단 생략')


## 11. Task 1·3용 예측 파일 출력

`PREDICT_SPLIT`의 공개 masked 입력으로 예측합니다. 패널별 출력은 `submissions/`에 저장됩니다. Task 3는 이 Task 1 결과를 사용하며 별도 Task 3 CSV를 만들지 않습니다. 전체 대회 업로드 형식으로 합칠 때는 주최 측 병합 절차를 적용해야 합니다. 이 노트북은 Task 2·4를 생성하지 않습니다.

In [ ]:
if PREDICT_SPLIT is not None:
    SUBMISSION_PATHS = predict_split(CONTEXTS, MODEL, TENSORS, SELECTED_ALPHA,
                                     PREDICT_SPLIT, REPORT_DIR / 'submissions')
    display(pd.DataFrame({'task1_task3_csv': SUBMISSION_PATHS}))
else:
    print('예측 CSV 생성 생략')
timing = dict(total_seconds=time.perf_counter() - RUN_STARTED,
              first_panel_time_to_score_seconds=FIRST_SCORE_SECONDS,
              selected_alpha=SELECTED_ALPHA, device=str(DEVICE))
(REPORT_DIR / 'timing.json').write_text(json.dumps(timing, indent=2))
print(timing)


## 참고 근거와 검증 범위

- [MagiNet: Mask-Aware Graph Imputation Network for Incomplete Traffic Data](https://arxiv.org/abs/2406.03511)
- [Colab 공식 GitHub 연동 안내](https://github.com/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)
- [MagiNet 공식 모델 구현](https://github.com/JeremyChou28/MagiNet/blob/main/models/model.py), [실행 구성](https://github.com/JeremyChou28/MagiNet/blob/main/main.py)
- [공식 Task 1 scorer](https://github.com/jacky850/trafficflowbench-public/blob/main/src/task1/score_task1.py)
- [공식 Task 3 scorer](https://github.com/jacky850/trafficflowbench-public/blob/main/src/task3/score_task3.py)

확인일: 2026-10-12. MagiNet의 핵심 구조를 독립적으로 재구현했고 대회용 채널·가변 패널·물리 보조 손실을 추가했습니다. 출처 코드의 고정 노드 행렬 및 고정 시간층과 동일한 가중치를 읽는 구현은 아닙니다.

파일 작성 시 수행한 검사는 노트북 형식·셀 문법·정의와 호출의 연결 및 데이터/마스크/단위 흐름에 대한 정적 검토입니다. 프로젝트 규칙에 따라 합성 입력 자체 검사, 실제 데이터 전처리, 학습, GPU 메모리, 시간, CSV 생성은 작성 환경에서 실행하지 않았습니다. 보조 손실·혼합이 고득점을 보장하지 않으며 위 실험으로 확인해야 합니다.
2026-10-12 개선: 보간 잔차 head, fit 관측 기반 입력 표준화와 버퍼 저장, 기존 NPZ ZIP 복원·재사용, 저장 과거 평균 및 캐시 계약 검증을 추가했습니다. 기존 절대값 모델과의 실제 성능 비교는 미실행입니다.
